In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

# ── AUTOMODEL FOR SEQUENCE CLASSIFICATION (distilbert-base-uncased-finetuned-sst-2-english) ──
# Task         : Sentiment classification — Positive / Negative
# Architecture : DistilBERT — distilled (smaller, faster) version of BERT
#                6 transformer layers vs BERT's 12, ~40% smaller, ~60% faster
# Head         : Linear layer on top of [CLS] token → 2 output logits (NEGATIVE, POSITIVE)
# Fine-tuned on: SST-2 (Stanford Sentiment Treebank) — movie review sentences
# Output       : raw logits → softmax → probabilities → argmax → label
# ─────────────────────────────────────────────────────────────────────────────────────────────

# ── STEP 1 : Load tokenizer ───────────────────────────────────────────────────
# Same tokenizer as used during fine-tuning — vocab and special tokens must match
# AutoTokenizer inspects config.json in the model repo → picks the right tokenizer class
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

print("—" * 60)
print("STEP 1 — Tokenizer loaded")
print(f"  Tokenizer class : {type(tokenizer).__name__}")
print(f"  Vocab size      : {tokenizer.vocab_size:,}")
print(f"  [CLS] id        : {tokenizer.cls_token_id}")
print(f"  [SEP] id        : {tokenizer.sep_token_id}")


# ── STEP 2 : Load model ───────────────────────────────────────────────────────
# AutoModelForSequenceClassification adds a classification head on top of the base model
# .from_pretrained() downloads:
#   config.json   → architecture details (layers, hidden size, num_labels, id2label)
#   pytorch_model.bin (or model.safetensors) → actual weights
# model.eval() → switches off dropout layers (training-only behaviour)
#   during training : dropout randomly zeros activations → regularisation
#   during inference: dropout must be OFF — you want deterministic, full output
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english"
)
model.eval()   # IMPORTANT — always call this before inference

print("\nSTEP 2 — Model loaded")
print(f"  Model class  : {type(model).__name__}")
print(f"  Num labels   : {model.config.num_labels}")
print(f"  id → label   : {model.config.id2label}")
#   id → label : {0: 'NEGATIVE', 1: 'POSITIVE'}


# ── STEP 3 : Tokenize input ───────────────────────────────────────────────────
# return_tensors="pt" → PyTorch tensors (not plain lists)
# produces: input_ids, attention_mask, (token_type_ids for BERT-based)
text = "This movie was absolutely fantastic!"
inputs = tokenizer(text, return_tensors="pt")

print("\nSTEP 3 — Input tokenized")
print(f"  Raw text       : {text!r}")
print(f"  input_ids      : {inputs['input_ids']}")
print(f"  attention_mask : {inputs['attention_mask']}")
print(f"  Tokens         : {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].tolist())}")
#   ['[CLS]', 'this', 'movie', 'was', 'absolutely', 'fantastic', '!', '[SEP]']


# ── STEP 4 : Forward pass ─────────────────────────────────────────────────────
# torch.no_grad() → disables gradient computation for all ops inside the block
#   during training  : gradients are tracked so .backward() can update weights
#   during inference : you never call .backward(), so tracking wastes memory
#   no_grad reduces memory by ~50% and speeds up the forward pass
# model(**inputs) unpacks the dict → same as model(input_ids=..., attention_mask=...)
with torch.no_grad():
    outputs = model(**inputs)

# outputs is a SequenceClassifierOutput dataclass — a named container, not a plain tensor
# .logits → raw scores before any activation function, shape [batch_size, num_labels]
#   logit[0] = score for NEGATIVE
#   logit[1] = score for POSITIVE
#   higher score = model is more confident about that class
#   logits can be any real number (negative, zero, positive) — NOT probabilities yet
logits = outputs.logits

print("\nSTEP 4 — Forward pass complete")
print(f"  Output type    : {type(outputs).__name__}")
print(f"  Logits shape   : {logits.shape}")       # torch.Size([1, 2])
print(f"  Raw logits     : {logits}")
#   tensor([[-3.4620,  3.6873]])  →  POSITIVE score >> NEGATIVE score


# ── STEP 5 : Post-process outputs ────────────────────────────────────────────
# softmax converts raw logits → probabilities that sum to 1.0
#   softmax(x_i) = exp(x_i) / sum(exp(x_j) for all j)
#   preserves relative order — highest logit → highest probability
#   dim=-1 → apply across last dimension (the num_labels axis)
probabilities = F.softmax(logits, dim=-1)

# argmax → index of the highest value → the predicted class id
predicted_class_id = logits.argmax(dim=-1).item()   # .item() → plain Python int

# model.config.id2label maps integer id → human-readable label string
predicted_label = model.config.id2label[predicted_class_id]

print("\nSTEP 5 — Post-processing")
print(f"  Probabilities  : {probabilities}")
#   tensor([[0.0006, 0.9994]])  →  0.06% NEGATIVE, 99.94% POSITIVE
print(f"  Predicted id   : {predicted_class_id}")   # 1
print(f"  Predicted label: {predicted_label}")       # POSITIVE

for class_id, label in model.config.id2label.items():
    print(f"    {label:10s} : {probabilities[0][class_id].item():.4f}")
#   NEGATIVE   : 0.0006
#   POSITIVE   : 0.9994


# ── AUTOMODEL VARIANTS — TASK REFERENCE ──────────────────────────────────────
# Class                                Head added on top of base model
# ─────────────────────────────────────────────────────────────────────────────
# AutoModel                            None — raw hidden states only
# AutoModelForSequenceClassification   Linear([CLS] → num_labels)
# AutoModelForTokenClassification      Linear(each token → num_labels) — NER, POS
# AutoModelForQuestionAnswering        Two linears — start logits + end logits per token
# AutoModelForMaskedLM                 Linear(each token → vocab_size) — fill [MASK]
# AutoModelForCausalLM                 Linear(each token → vocab_size) — next token
# AutoModelForSeq2SeqLM                Encoder + Decoder heads — translation, summarise
# AutoModelForMultipleChoice           Linear([CLS] per choice → 1 score) — MCQ
# ─────────────────────────────────────────────────────────────────────────────
# Rule : pick the class that matches your task
#        the head is the only part that changes — base transformer stays the same


# ── OUTPUT DATACLASS FIELDS — WHAT EACH VARIANT RETURNS ──────────────────────
# SequenceClassifierOutput   → .logits                    shape [B, num_labels]
# TokenClassifierOutput      → .logits                    shape [B, seq_len, num_labels]
# QuestionAnsweringOutput    → .start_logits, .end_logits shape [B, seq_len] each
# MaskedLMOutput             → .logits                    shape [B, seq_len, vocab_size]
# CausalLMOutput             → .logits                    shape [B, seq_len, vocab_size]
# BaseModelOutput            → .last_hidden_state         shape [B, seq_len, hidden_size]
# All variants optionally return:
#   .hidden_states  → tuple of tensors, one per layer (needs output_hidden_states=True)
#   .attentions     → tuple of attention weight matrices  (needs output_attentions=True)


# ── DRY RUN : text = "This movie was absolutely fantastic!" ──────────────────

# Stage 1 — Lowercase
# "This movie was absolutely fantastic!"
#  → "this movie was absolutely fantastic!"

# Stage 2 — tokenizer.tokenize() : WordPiece split, no specials, no IDs
# word          pieces              reason
# this       →  this               whole word in vocab
# movie      →  movie              whole word in vocab
# was        →  was                whole word in vocab
# absolutely →  absolutely         whole word in vocab
# fantastic  →  fantastic          whole word in vocab
# !          →  !                  punctuation split
#
# tokens = ['this', 'movie', 'was', 'absolutely', 'fantastic', '!']

# Stage 3 — tokenizer() : add specials + IDs
# pos  token        id
#  0   [CLS]       101
#  1   this        2023
#  2   movie       3185
#  3   was         2001
#  4   absolutely  7078
#  5   fantastic   10392
#  6   !            999
#  7   [SEP]        102
#
# input_ids      = tensor([[101, 2023, 3185, 2001, 7078, 10392, 999, 102]])
# attention_mask = tensor([[  1,    1,    1,    1,    1,     1,   1,   1]])

# Stage 4 — Forward pass (simplified)
# [CLS] token walks through 6 transformer layers
# each layer : self-attention → add & norm → feed-forward → add & norm
# after 6 layers: [CLS] hidden state shape = [1, 768]  (hidden_size = 768)
# classification head : Linear(768 → 2) applied to [CLS] hidden state
# raw logits = [[-3.4620, 3.6873]]
#               NEGATIVE  POSITIVE   → POSITIVE wins by large margin

# Stage 5 — Post-processing
# softmax([-3.4620, 3.6873])
#   exp(-3.4620) = 0.0314
#   exp( 3.6873) = 39.93
#   sum          = 39.96
#   probabilities = [0.0314/39.96,  39.93/39.96]
#                 = [0.0006,        0.9994]
# argmax         = 1   → id2label[1] = "POSITIVE"
# final answer   : POSITIVE  (99.94% confidence)

————————————————————————————————————————————————————————————
STEP 1 — Tokenizer loaded
  Tokenizer class : BertTokenizer
  Vocab size      : 30,522
  [CLS] id        : 101
  [SEP] id        : 102


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


STEP 2 — Model loaded
  Model class  : DistilBertForSequenceClassification
  Num labels   : 2
  id → label   : {0: 'NEGATIVE', 1: 'POSITIVE'}

STEP 3 — Input tokenized
  Raw text       : 'This movie was absolutely fantastic!'
  input_ids      : tensor([[  101,  2023,  3185,  2001,  7078, 10392,   999,   102]])
  attention_mask : tensor([[1, 1, 1, 1, 1, 1, 1, 1]])
  Tokens         : ['[CLS]', 'this', 'movie', 'was', 'absolutely', 'fantastic', '!', '[SEP]']

STEP 4 — Forward pass complete
  Output type    : SequenceClassifierOutput
  Logits shape   : torch.Size([1, 2])
  Raw logits     : tensor([[-4.3171,  4.6654]])

STEP 5 — Post-processing
  Probabilities  : tensor([[1.2557e-04, 9.9987e-01]])
  Predicted id   : 1
  Predicted label: POSITIVE
    NEGATIVE   : 0.0001
    POSITIVE   : 0.9999


In [4]:
from transformers import AutoTokenizer, AutoModel
import torch

# ── AUTOMODEL — RAW EMBEDDINGS (bert-base-uncased) ───────────────────────────
# Task         : No task — returns raw contextual representations only
# Architecture : BERT-base — 12 transformer layers, 12 attention heads
# Hidden size  : 768 — every token gets a 768-dimensional vector
# No head      : AutoModel has NO linear layer on top
#                → you get the full transformer body output, nothing more
#                → compare: AutoModelForSequenceClassification adds Linear(768→2)
# Use when     : you want embeddings for similarity, clustering, fine-tuning,
#                retrieval, or any custom downstream task
# Outputs      : .last_hidden_state → contextual vector per token
#                .pooler_output     → [CLS] vector through a Linear + Tanh
#                .hidden_states     → every layer's output (if requested)
#                .attentions        → every layer's attention weights (if requested)
# ─────────────────────────────────────────────────────────────────────────────

# ── STEP 1 : Load tokenizer ───────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("─" * 60)
print("STEP 1 — Tokenizer loaded")
print(f"  Tokenizer class : {type(tokenizer).__name__}")
print(f"  Vocab size      : {tokenizer.vocab_size:,}")


# ── STEP 2 : Load model ───────────────────────────────────────────────────────
# attn_implementation="eager" — use classic attention instead of torch sdpa
# sdpa (scaled dot-product attention) — faster, fused kernel, but cannot return
#   attention weight matrices because the fused op discards them mid-computation
# eager — computes attention step by step, keeps all intermediate tensors
# ONLY needed when output_attentions=True — for all other steps sdpa is fine
# two separate model objects so normal inference stays fast (sdpa)
# and attention inspection uses eager only where required
model      = AutoModel.from_pretrained("bert-base-uncased")
model.eval()

model_eager = AutoModel.from_pretrained(
    "bert-base-uncased",
    attn_implementation="eager"   # ← required for output_attentions=True
)
model_eager.eval()

print("\nSTEP 2 — Models loaded")
print(f"  model class       : {type(model).__name__}")
print(f"  attn (normal)     : sdpa  — fast, no attention weights")
print(f"  attn (eager)      : eager — slower, returns attention weights")
print(f"  Num layers        : {model.config.num_hidden_layers}")
print(f"  Hidden size       : {model.config.hidden_size}")
print(f"  Num heads         : {model.config.num_attention_heads}")
# UNEXPECTED keys warning — safe to ignore
#   bert-base-uncased checkpoint contains MLM + NSP head weights
#   (cls.predictions.*, cls.seq_relationship.*)
#   AutoModel loads only the transformer body — no heads
#   those weights have nowhere to go → flagged as UNEXPECTED
#   only a problem if you expected the full MLM model (use AutoModelForMaskedLM then)


# ── STEP 3 : Tokenize input ───────────────────────────────────────────────────
text   = "Hello world"
inputs = tokenizer(text, return_tensors="pt")

print("\nSTEP 3 — Input tokenized")
print(f"  Raw text       : {text!r}")
print(f"  input_ids      : {inputs['input_ids']}")
print(f"  attention_mask : {inputs['attention_mask']}")
print(f"  Tokens         : {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].tolist())}")
#   ['[CLS]', 'hello', 'world', '[SEP]']


# ── STEP 4 : Forward pass ─────────────────────────────────────────────────────
# torch.no_grad() — no gradient tracking, saves memory, speeds up inference
with torch.no_grad():
    outputs = model(**inputs)

print("\nSTEP 4 — Forward pass complete")
print(f"  Output type : {type(outputs).__name__}")


# ── STEP 5A : last_hidden_state ───────────────────────────────────────────────
# Shape — [batch_size, seq_len, hidden_size]
# One 768-dim vector per token, fully context-aware
# Every vector "knows" about all other tokens via self-attention
# [CLS] at index 0, [SEP] at index -1
last_hidden = outputs.last_hidden_state

print("\nSTEP 5A — last_hidden_state")
print(f"  Shape              : {last_hidden.shape}")
#   torch.Size([1, 4, 768])   — batch=1, tokens=4 ([CLS] hello world [SEP])
print(f"  [CLS] vector[:5]   : {last_hidden[0, 0, :5]}")
print(f"  'hello' vector[:5] : {last_hidden[0, 1, :5]}")
print(f"  'world' vector[:5] : {last_hidden[0, 2, :5]}")


# ── STEP 5B : pooler_output ───────────────────────────────────────────────────
# Takes the [CLS] hidden state (index 0) → passes through Linear(768→768) + Tanh
# Designed during BERT pre-training for next-sentence prediction
# One vector per sentence — shape [batch_size, hidden_size]
# NOTE — for sentence similarity tasks, mean pooling (below) usually works better
pooler = outputs.pooler_output

print("\nSTEP 5B — pooler_output")
print(f"  Shape          : {pooler.shape}")
#   torch.Size([1, 768])
print(f"  Vector[:5]     : {pooler[0, :5]}")


# ── STEP 5C : mean pooling (sentence embedding) ───────────────────────────────
# Why not just use pooler_output?
#   pooler_output uses only [CLS] — loses information from all other tokens
#   mean pooling averages ALL real token vectors → richer sentence representation
# Why use attention_mask?
#   padding tokens (mask=0) must be excluded from the average
#   without mask, padding zeros would drag all vectors toward zero
# Steps:
#   1. unsqueeze(-1)   — expand mask from [B, seq] to [B, seq, 1]
#   2. .expand(...)    — broadcast to [B, seq, 768] to match token_embeddings
#   3. multiply        — zero out padding positions in the embedding matrix
#   4. sum(dim=1)      — sum across token axis → [B, 768]
#   5. clamp + divide  — divide by real token count, clamp avoids division by zero
attention_mask   = inputs["attention_mask"]
token_embeddings = outputs.last_hidden_state
mask_expanded    = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
sentence_embedding = (
    torch.sum(token_embeddings * mask_expanded, dim=1)
    / torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
)

print("\nSTEP 5C — Mean pooling (sentence embedding)")
print(f"  attention_mask shape   : {attention_mask.shape}")
#   torch.Size([1, 4])
print(f"  mask_expanded shape    : {mask_expanded.shape}")
#   torch.Size([1, 4, 768])
print(f"  sentence_embedding shape : {sentence_embedding.shape}")
#   torch.Size([1, 768])
print(f"  Vector[:5]             : {sentence_embedding[0, :5]}")


# ── STEP 5D : all hidden states (every layer) ─────────────────────────────────
# output_hidden_states=True — returns output of every transformer layer
# .hidden_states — tuple of (num_layers + 1) tensors
#   index 0 — embedding layer output (token + position + segment embeddings summed)
#   index 1 — layer 1 output
#   ...
#   index 12 — layer 12 output (same as last_hidden_state)
# Each tensor shape — [batch_size, seq_len, hidden_size]
# Use case — layer-wise analysis, probing tasks, weighted-layer embeddings
with torch.no_grad():
    outputs_hidden = model(**inputs, output_hidden_states=True)

all_hidden = outputs_hidden.hidden_states

print("\nSTEP 5D — All hidden states")
print(f"  Num tensors in tuple : {len(all_hidden)}")
#   13  — embedding layer + 12 transformer layers
print(f"  Each tensor shape    : {all_hidden[0].shape}")
#   torch.Size([1, 4, 768])
for i, h in enumerate(all_hidden):
    label = "embedding" if i == 0 else f"layer {i:>2}"
    print(f"    {label} — {h.shape}")


# ── STEP 5E : attention weights (every layer, every head) ─────────────────────
# use model_eager here — sdpa model will return empty tuple ()
with torch.no_grad():
    outputs_attn = model_eager(**inputs, output_attentions=True)

all_attentions = outputs_attn.attentions

print("\nSTEP 5E — All attention weights")
print(f"  Num tensors in tuple : {len(all_attentions)}")
#   12 — one per transformer layer
print(f"  Each tensor shape    : {all_attentions[0].shape}")
#   torch.Size([1, 12, 4, 4]) — batch=1, heads=12, seq=4, seq=4
print(f"  Layer 1 head 0 attention matrix:")
print(f"  {all_attentions[0][0, 0]}")
#   each row sums to ~1.0 — softmax applied across key axis
for i, attn in enumerate(all_attentions):
    print(f"    layer {i+1:>2} — {attn.shape}")


# ── OUTPUT FIELDS — QUICK REFERENCE ──────────────────────────────────────────
# outputs.last_hidden_state   shape [B, seq_len, 768]  — always returned
# outputs.pooler_output       shape [B, 768]           — always returned
# outputs.hidden_states       tuple of 13 × [B, seq_len, 768]  — opt-in
# outputs.attentions          tuple of 12 × [B, heads, seq, seq] — opt-in
# ─────────────────────────────────────────────────────────────────────────────
# Choosing your embedding:
#   single token task  →  last_hidden_state[:, token_index, :]
#   sentence (fast)    →  pooler_output
#   sentence (better)  →  mean pooling over last_hidden_state
#   layer analysis     →  hidden_states[layer_index]
#   interpretability   →  attentions[layer_index]


# ── DRY RUN : text = "Hello world" ───────────────────────────────────────────

# Stage 1 — Tokenize
# "Hello world"
#  → lowercase → "hello world"
#  → WordPiece → ['hello', 'world']  (both in vocab, no subword splitting)
#  → add specials → ['[CLS]', 'hello', 'world', '[SEP]']
#  → IDs         → [101, 7592, 2088, 102]
#
# input_ids      = tensor([[101, 7592, 2088, 102]])
# attention_mask = tensor([[  1,    1,    1,   1]])    — no padding, all real tokens

# Stage 2 — Forward pass (simplified, one layer shown)
# Each of the 12 layers runs:
#   self-attention  — each token looks at all others, weighted by relevance
#   add & norm      — residual connection + LayerNorm for stability
#   feed-forward    — two linear layers with GELU activation
#   add & norm      — residual + LayerNorm again
# After layer 12:
#   last_hidden_state shape — [1, 4, 768]
#     index 0 → [CLS]   768-dim vector (context of full sentence)
#     index 1 → hello   768-dim vector (contextualised by "world")
#     index 2 → world   768-dim vector (contextualised by "hello")
#     index 3 → [SEP]   768-dim vector

# Stage 3 — pooler_output
# takes last_hidden_state[0, 0, :]   →   [CLS] vector, shape [768]
# → Linear(768 → 768) → Tanh
# → pooler_output shape [1, 768]

# Stage 4 — mean pooling
# mask_expanded zeros out nothing (no padding here — all mask values = 1)
# sum([CLS] + hello + world + [SEP]) / 4
# → sentence_embedding shape [1, 768]
# in practice [CLS] and [SEP] add noise — for cleaner embeddings strip them:
#   token_embeddings[:, 1:-1, :]  before mean pooling

────────────────────────────────────────────────────────────
STEP 1 — Tokenizer loaded
  Tokenizer class : BertTokenizer
  Vocab size      : 30,522


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



STEP 2 — Models loaded
  model class       : BertModel
  attn (normal)     : sdpa  — fast, no attention weights
  attn (eager)      : eager — slower, returns attention weights
  Num layers        : 12
  Hidden size       : 768
  Num heads         : 12

STEP 3 — Input tokenized
  Raw text       : 'Hello world'
  input_ids      : tensor([[ 101, 7592, 2088,  102]])
  attention_mask : tensor([[1, 1, 1, 1]])
  Tokens         : ['[CLS]', 'hello', 'world', '[SEP]']

STEP 4 — Forward pass complete
  Output type : BaseModelOutputWithPoolingAndCrossAttentions

STEP 5A — last_hidden_state
  Shape              : torch.Size([1, 4, 768])
  [CLS] vector[:5]   : tensor([-0.1689,  0.1361, -0.1394, -0.0544, -0.2953])
  'hello' vector[:5] : tensor([-0.3633,  0.1412,  0.8800, -1.0147, -0.1198])
  'world' vector[:5] : tensor([-0.6986, -0.6988,  0.0645, -0.4830,  0.1972])

STEP 5B — pooler_output
  Shape          : torch.Size([1, 768])
  Vector[:5]     : tensor([-0.9062, -0.3112, -0.6217,  0.7741,  0.2899]

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

# ── AUTOMODEL FOR SEQUENCE CLASSIFICATION (distilbert-base-uncased-finetuned-sst-2-english) ──
# Task         : Single-label text classification — one label per sentence
# Architecture : DistilBERT — 6 transformer layers, distilled from BERT's 12
#                ~40% fewer parameters, ~60% faster, ~97% of BERT's performance
# Head         : Linear(768 → num_labels) applied to [CLS] token only
#                [CLS] aggregates meaning of full sequence through self-attention
#                all other token outputs are discarded after the transformer body
# Fine-tuned on: SST-2 — Stanford Sentiment Treebank, movie review sentences
# num_labels   : 2 — NEGATIVE (0), POSITIVE (1)
# Output       : raw logits → softmax → probabilities → argmax → label string
# Use for      : sentiment analysis, topic classification, intent detection, spam
# ─────────────────────────────────────────────────────────────────────────────────────────────

# ── STEP 1 : Load tokenizer ───────────────────────────────────────────────────
# must match the tokenizer used during fine-tuning — vocab and special tokens must align
# AutoTokenizer reads config.json in the model repo → selects the correct class
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

print("─" * 60)
print("STEP 1 — Tokenizer loaded")
print(f"  Tokenizer class : {type(tokenizer).__name__}")
print(f"  Vocab size      : {tokenizer.vocab_size:,}")
print(f"  [CLS] id        : {tokenizer.cls_token_id}")
print(f"  [SEP] id        : {tokenizer.sep_token_id}")


# ── STEP 2 : Load model ───────────────────────────────────────────────────────
# AutoModelForSequenceClassification — transformer body + classification head
# head — Linear(hidden_size → num_labels) on top of [CLS] hidden state
# model.eval() — disables dropout for deterministic inference output
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english"
)
model.eval()

print("\nSTEP 2 — Model loaded")
print(f"  Model class  : {type(model).__name__}")
print(f"  Num labels   : {model.config.num_labels}")
print(f"  id2label     : {model.config.id2label}")
#   {0: 'NEGATIVE', 1: 'POSITIVE'}
print(f"  label2id     : {model.config.label2id}")
#   {'NEGATIVE': 0, 'POSITIVE': 1}


# ── STEP 3 : Tokenize input ───────────────────────────────────────────────────
# return_tensors="pt" — PyTorch tensors, required for model(**inputs)
# produces: input_ids, attention_mask
# DistilBERT does NOT use token_type_ids (no segment embeddings)
text   = "I love this product!"
inputs = tokenizer(text, return_tensors="pt")

print("\nSTEP 3 — Input tokenized")
print(f"  Raw text       : {text!r}")
print(f"  input_ids      : {inputs['input_ids']}")
print(f"  attention_mask : {inputs['attention_mask']}")
print(f"  Tokens         : {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].tolist())}")
#   ['[CLS]', 'i', 'love', 'this', 'product', '!', '[SEP]']


# ── STEP 4 : Forward pass ─────────────────────────────────────────────────────
# torch.no_grad() — disables gradient computation
#   training   : gradients tracked → .backward() updates weights
#   inference  : gradients never needed → tracking wastes memory and compute
# model(**inputs) unpacks dict → model(input_ids=..., attention_mask=...)
# internally:
#   input_ids → embedding layer → 6 transformer layers → [CLS] hidden state
#   [CLS] hidden state → Linear(768 → 2) → raw logits
with torch.no_grad():
    outputs = model(**inputs)

print("\nSTEP 4 — Forward pass complete")
print(f"  Output type  : {type(outputs).__name__}")
#   SequenceClassifierOutput


# ── STEP 5 : Post-process — single-label classification ──────────────────────
# logits — raw scores, shape [batch_size, num_labels]
#   NOT probabilities — no activation applied yet
#   can be any real number (negative, zero, positive)
#   higher logit = model leans more toward that class
#   logit[0] = NEGATIVE score, logit[1] = POSITIVE score
logits = outputs.logits

print("\nSTEP 5 — Post-processing (single-label)")
print(f"  Logits shape   : {logits.shape}")
#   torch.Size([1, 2])
print(f"  Raw logits     : {logits}")
#   tensor([[-3.9, 4.1]])   — POSITIVE score dominates by large margin

# softmax — converts logits to probabilities that sum to 1.0
#   softmax(x_i) = exp(x_i) / Σ exp(x_j)
#   preserves relative order — highest logit → highest probability
#   use softmax for single-label (exactly one class is correct)
probs = F.softmax(logits, dim=-1)
print(f"  Probabilities  : {probs}")
#   tensor([[0.0002, 0.9998]])

# argmax — index of highest value → predicted class id
# .item() — converts 0-dim tensor to plain Python int
pred_id = logits.argmax(dim=-1).item()
label   = model.config.id2label[pred_id]
print(f"  Predicted id   : {pred_id}")
#   1
print(f"  Predicted label: {label}")
#   POSITIVE

print("\n  Per-class breakdown:")
for class_id, class_label in model.config.id2label.items():
    print(f"    {class_label:10s} — {probs[0][class_id].item():.4f}")
#   NEGATIVE   — 0.0002
#   POSITIVE   — 0.9998


# ── STEP 6 : Multi-label classification ──────────────────────────────────────
# single-label  — exactly one class is correct     → softmax (probs sum to 1.0)
# multi-label   — multiple classes can be correct  → sigmoid (each prob independent)
#
# sigmoid — squashes each logit independently to [0, 1]
#   sigmoid(x) = 1 / (1 + exp(-x))
#   each output is its own binary probability — unrelated to other outputs
#   no constraint that they sum to 1.0
#   threshold at 0.5 → above = class present, below = class absent
#
# example: topic tagging — a sentence can be [sports=0.9, news=0.8, politics=0.1]
# softmax would force them to compete — sigmoid lets each be independent
probs_multilabel = torch.sigmoid(logits)

print("\nSTEP 6 — Multi-label sigmoid (for reference)")
print(f"  sigmoid logits : {probs_multilabel}")
#   not meaningful here (SST-2 is single-label) — shown for format only
#   for real multi-label: classes above 0.5 threshold are predicted present
print(f"  Threshold 0.5  :")
for class_id, class_label in model.config.id2label.items():
    present = probs_multilabel[0][class_id].item() > 0.5
    print(f"    {class_label:10s} — {probs_multilabel[0][class_id].item():.4f}  →  {'present' if present else 'absent'}")


# ── SINGLE-LABEL vs MULTI-LABEL — DECISION GUIDE ─────────────────────────────
# single-label  : one correct answer per input   →  softmax + argmax
#   sentiment   : POSITIVE or NEGATIVE (not both)
#   intent      : book_flight or check_weather (not both)
#   topic       : sports or politics (mutually exclusive)
#
# multi-label   : multiple correct answers per input  →  sigmoid + threshold
#   tags        : a news article can be sports AND politics AND local
#   emotions    : a sentence can be sad AND angry simultaneously
#   genres      : a movie can be action AND comedy AND thriller
# ─────────────────────────────────────────────────────────────────────────────


# ── DRY RUN : text = "I love this product!" ──────────────────────────────────

# Stage 1 — Tokenize
# "I love this product!"
#  → lowercase     → "i love this product!"
#  → WordPiece     → ['i', 'love', 'this', 'product', '!']  (all in vocab)
#  → add specials  → ['[CLS]', 'i', 'love', 'this', 'product', '!', '[SEP]']
#  → IDs           → [101, 1045, 2293, 2023, 4031, 999, 102]
#
# input_ids      = tensor([[101, 1045, 2293, 2023, 4031, 999, 102]])
# attention_mask = tensor([[  1,    1,    1,    1,    1,   1,   1]])   — no padding

# Stage 2 — Forward pass (simplified)
# token IDs → embedding layer → 6 DistilBERT transformer layers
# each layer: self-attention → add & norm → feed-forward → add & norm
# after layer 6: [CLS] hidden state shape [1, 768]
# classification head: Linear(768 → 2) applied to [CLS] only
# raw logits ≈ [[-3.9, 4.1]]
#               NEGATIVE  POSITIVE  → POSITIVE dominates

# Stage 3 — Post-processing
# softmax([-3.9, 4.1])
#   exp(-3.9) = 0.0202
#   exp( 4.1) = 60.34
#   sum        = 60.36
#   probs      = [0.0202/60.36, 60.34/60.36]
#              = [0.0002,       0.9998]
# argmax       = 1  →  id2label[1] = "POSITIVE"
# final answer : POSITIVE  (99.98% confidence)

────────────────────────────────────────────────────────────
STEP 1 — Tokenizer loaded
  Tokenizer class : BertTokenizer
  Vocab size      : 30,522
  [CLS] id        : 101
  [SEP] id        : 102


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


STEP 2 — Model loaded
  Model class  : DistilBertForSequenceClassification
  Num labels   : 2
  id2label     : {0: 'NEGATIVE', 1: 'POSITIVE'}
  label2id     : {'NEGATIVE': 0, 'POSITIVE': 1}

STEP 3 — Input tokenized
  Raw text       : 'I love this product!'
  input_ids      : tensor([[ 101, 1045, 2293, 2023, 4031,  999,  102]])
  attention_mask : tensor([[1, 1, 1, 1, 1, 1, 1]])
  Tokens         : ['[CLS]', 'i', 'love', 'this', 'product', '!', '[SEP]']

STEP 4 — Forward pass complete
  Output type  : SequenceClassifierOutput

STEP 5 — Post-processing (single-label)
  Logits shape   : torch.Size([1, 2])
  Raw logits     : tensor([[-4.3599,  4.7156]])
  Probabilities  : tensor([[1.1443e-04, 9.9989e-01]])
  Predicted id   : 1
  Predicted label: POSITIVE

  Per-class breakdown:
    NEGATIVE   — 0.0001
    POSITIVE   — 0.9999

STEP 6 — Multi-label sigmoid (for reference)
  sigmoid logits : tensor([[0.0126, 0.9911]])
  Threshold 0.5  :
    NEGATIVE   — 0.0126  →  absent
    POSITIVE   — 0.99

In [6]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
import torch.nn.functional as F

# ── AUTOMODEL FOR TOKEN CLASSIFICATION (dslim/bert-base-NER) ─────────────────
# Task         : Named Entity Recognition — label every token individually
# Architecture : BERT-base — 12 transformer layers, 12 attention heads, hidden 768
# Head         : Linear(768 → num_labels) applied to EVERY token (not just [CLS])
#                each token gets its own logit vector → its own label prediction
#                contrast with SequenceClassification — only [CLS] goes to head
# Fine-tuned on: CoNLL-2003 NER dataset — newswire text with 4 entity types
# num_labels   : 9 — O + B/I pairs for PER, ORG, LOC, MISC
# IOB2 scheme  : B- = beginning of entity span
#                I- = inside / continuation of entity span
#                O  = outside — not part of any entity
# Use for      : NER, POS tagging, chunking, any per-token labeling task
# ─────────────────────────────────────────────────────────────────────────────

# ── STEP 1 : Load tokenizer ───────────────────────────────────────────────────
# dslim/bert-base-NER was fine-tuned starting from bert-base-cased
# cased — capital letters preserved — critical for NER
# "Sarah" ≠ "sarah" — capitalisation is a strong signal for proper nouns
# using uncased here would hurt entity detection significantly
tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")

print("─" * 60)
print("STEP 1 — Tokenizer loaded")
print(f"  Tokenizer class : {type(tokenizer).__name__}")
print(f"  Vocab size      : {tokenizer.vocab_size:,}")
print(f"  Cased           : {not tokenizer.do_lower_case}")
#   Cased : True — capital letters preserved, essential for NER


# ── STEP 2 : Load model ───────────────────────────────────────────────────────
# AutoModelForTokenClassification — transformer body + per-token linear head
# head — Linear(768 → num_labels) applied independently to every token position
# model.eval() — disables dropout for deterministic inference
model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")
model.eval()

print("\nSTEP 2 — Model loaded")
print(f"  Model class  : {type(model).__name__}")
print(f"  Num labels   : {model.config.num_labels}")
print(f"  id2label     : {model.config.id2label}")
#   {0: 'O', 1: 'B-MISC', 2: 'I-MISC', 3: 'B-PER', 4: 'I-PER',
#    5: 'B-ORG', 6: 'I-ORG', 7: 'B-LOC', 8: 'I-LOC'}
print(f"  label2id     : {model.config.label2id}")


# ── STEP 3 : Tokenize input ───────────────────────────────────────────────────
# cased model — preserves capitalisation exactly as written
# WordPiece may split rare words into subwords
#   e.g. "London" → ["London"]      (in vocab, no split)
#   e.g. "Schwarzenegger" → ["Sc", "##hwar", "##zen", "##eg", "##ger"]
# subword alignment problem — one word may produce multiple tokens
#   the model assigns a label to EACH subword token
#   common fix: take label of first subword, ignore ##-prefixed continuations
text   = "My name is Sarah and I live in London."
inputs = tokenizer(text, return_tensors="pt")

print("\nSTEP 3 — Input tokenized")
print(f"  Raw text       : {text!r}")
print(f"  input_ids      : {inputs['input_ids']}")
print(f"  attention_mask : {inputs['attention_mask']}")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(f"  Tokens         : {tokens}")
print(f"  Num tokens     : {len(tokens)}")
#   ['[CLS]', 'My', 'name', 'is', 'Sarah', 'and', 'I', 'live', 'in', 'London', '.', '[SEP]']
#   12 tokens total — [CLS] + 10 word tokens + . + [SEP]


# ── STEP 4 : Forward pass ─────────────────────────────────────────────────────
# every token passes through all 12 BERT layers
# each layer: self-attention → add & norm → feed-forward → add & norm
# after layer 12: hidden state shape [batch, seq_len, 768]
# classification head: Linear(768 → 9) applied to EVERY position
# result: logits shape [batch, seq_len, num_labels]
#         one 9-dim logit vector per token — one label predicted per token
with torch.no_grad():
    outputs = model(**inputs)

print("\nSTEP 4 — Forward pass complete")
print(f"  Output type  : {type(outputs).__name__}")
#   TokenClassifierOutput


# ── STEP 5 : Post-process — per-token labels ──────────────────────────────────
# logits shape — [batch_size, seq_len, num_labels]
#   axis 0 — batch (1 sentence here)
#   axis 1 — token position (12 tokens here)
#   axis 2 — label scores (9 NER labels)
#   each token has its own independent 9-dim logit vector
logits = outputs.logits

print("\nSTEP 5 — Post-processing")
print(f"  Logits shape  : {logits.shape}")
#   torch.Size([1, 12, 9])  — batch=1, seq=12, labels=9

# argmax along label axis → predicted class id for each token position
# [0] — first (only) sentence in batch
pred_ids = logits.argmax(dim=-1)[0]
print(f"  Pred id shape : {pred_ids.shape}")
#   torch.Size([12])  — one predicted id per token

# softmax along label axis → confidence score for predicted label
probs = F.softmax(logits, dim=-1)[0]

print("\n  Token — label predictions:")
print(f"  {'Token':15} {'Label':10} {'Confidence':>10}")
print(f"  {'─'*15} {'─'*10} {'─'*10}")
for token, pred_id, prob_vec in zip(tokens, pred_ids, probs):
    label      = model.config.id2label[pred_id.item()]
    confidence = prob_vec[pred_id].item()
    print(f"  {token:15} {label:10} {confidence:>10.4f}")
# [CLS]           O          0.9998
# My              O          0.9991
# name            O          0.9995
# is              O          0.9994
# Sarah           B-PER      0.9986     ← beginning of person entity
# and             O          0.9997
# I               O          0.9961
# live            O          0.9996
# in              O          0.9992
# London          B-LOC      0.9979     ← beginning of location entity
# .               O          0.9998
# [SEP]           O          0.9997


# ── STEP 6 : Entity span reconstruction ──────────────────────────────────────
# raw output gives one label per token — need to group into full entity spans
# IOB2 rules:
#   B-XXX → start a new entity of type XXX
#   I-XXX → continue the current entity (must follow B-XXX or I-XXX of same type)
#   O     → not part of any entity, close current span if one is open
# subword tokens (##-prefixed) inherit their word's label — skip them here
entities   = []
current    = None   # holds {"text": ..., "label": ..., "tokens": [...]}

for token, pred_id in zip(tokens, pred_ids):
    label = model.config.id2label[pred_id.item()]

    # skip special tokens — [CLS] and [SEP] are never real entities
    if token in ("[CLS]", "[SEP]"):
        continue

    # skip subword continuations — label belongs to the root token
    if token.startswith("##"):
        if current:
            current["tokens"].append(token[2:])   # strip ## and append to current word
        continue

    if label.startswith("B-"):
        # close previous entity if one was open
        if current:
            entities.append(current)
        # open new entity span
        current = {"text": token, "label": label[2:], "tokens": [token]}

    elif label.startswith("I-") and current and label[2:] == current["label"]:
        # continue current entity span — same type required
        current["tokens"].append(token)
        current["text"] = " ".join(current["tokens"])

    else:
        # O label or mismatched I- → close current span
        if current:
            entities.append(current)
            current = None

# close any entity still open at end of sequence
if current:
    entities.append(current)

print("\nSTEP 6 — Reconstructed entity spans:")
print(f"  {'Entity text':20} {'Type':10}")
print(f"  {'─'*20} {'─'*10}")
for ent in entities:
    print(f"  {ent['text']:20} {ent['label']:10}")
#   Sarah                PER
#   London               LOC


# ── IOB2 LABEL SCHEME — REFERENCE ────────────────────────────────────────────
# O        — Outside — not part of any named entity
# B-PER    — Beginning of a Person entity
# I-PER    — Inside / continuation of a Person entity
# B-ORG    — Beginning of an Organisation entity
# I-ORG    — Inside an Organisation entity
# B-LOC    — Beginning of a Location entity
# I-LOC    — Inside a Location entity
# B-MISC   — Beginning of a Miscellaneous entity (nationality, event, product)
# I-MISC   — Inside a Miscellaneous entity
# ─────────────────────────────────────────────────────────────────────────────
# Multi-token entity example:
#   "New  York  City  is  great"
#    B-LOC I-LOC I-LOC O   O
#   → one entity: "New York City" of type LOC
#   B starts the span — I continues it — O closes it


# ── SEQUENCE vs TOKEN CLASSIFICATION — KEY DIFFERENCE ─────────────────────────
# AutoModelForSequenceClassification
#   head applied to [CLS] token only → one label per sentence
#   logits shape — [batch, num_labels]
#
# AutoModelForTokenClassification
#   head applied to EVERY token → one label per token
#   logits shape — [batch, seq_len, num_labels]
#   argmax over dim=-1 to get predicted id per position
# ─────────────────────────────────────────────────────────────────────────────


# ── DRY RUN : text = "My name is Sarah and I live in London." ─────────────────

# Stage 1 — Tokenize (cased — no lowercasing)
# "My name is Sarah and I live in London."
# word       →  token(s)      reason
# My         →  My            in vocab, cased
# name       →  name          in vocab
# is         →  is            in vocab
# Sarah      →  Sarah         in vocab, proper noun — capitalisation preserved
# and        →  and           in vocab
# I          →  I             in vocab
# live       →  live          in vocab
# in         →  in            in vocab
# London     →  London        in vocab, proper noun
# .          →  .             punctuation split
#
# final tokens with specials:
# ['[CLS]', 'My', 'name', 'is', 'Sarah', 'and', 'I', 'live', 'in', 'London', '.', '[SEP]']
# input_ids shape      — [1, 12]
# attention_mask shape — [1, 12]   all 1s, no padding

# Stage 2 — Forward pass
# each token walks through 12 BERT layers (self-attention + FFN each layer)
# output: last_hidden_state shape [1, 12, 768] — one 768-dim vector per token
# classification head: Linear(768 → 9) applied at every position
# logits shape: [1, 12, 9] — one 9-dim score vector per token

# Stage 3 — argmax over label axis
# token       logits (simplified)               argmax → label
# [CLS]       [3.1, -1.2, -1.1, ...]            0 → O
# My          [2.8, -0.9, -0.8, ...]            0 → O
# name        [3.0, -1.0, -0.9, ...]            0 → O
# is          [2.9, -1.1, -1.0, ...]            0 → O
# Sarah       [-2.1, -1.3, -0.8, 4.2, ...]      3 → B-PER
# and         [3.1, -0.9, -0.8, ...]            0 → O
# I           [2.7, -0.7, -0.6, ...]            0 → O
# live        [3.2, -1.0, -0.9, ...]            0 → O
# in          [3.0, -1.1, -1.0, ...]            0 → O
# London      [-1.8, -0.9, -0.7, -1.1, ..4.5.] 7 → B-LOC
# .           [3.2, -1.0, -0.9, ...]            0 → O
# [SEP]       [3.0, -0.9, -0.8, ...]            0 → O

# Stage 4 — Entity reconstruction
# Sarah → B-PER → open span {type: PER}
# and   → O     → close span → entity: "Sarah" PER
# London → B-LOC → open span {type: LOC}
# .     → O     → close span → entity: "London" LOC
# final entities: [("Sarah", PER), ("London", LOC)]

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

────────────────────────────────────────────────────────────
STEP 1 — Tokenizer loaded
  Tokenizer class : BertTokenizer
  Vocab size      : 28,996
  Cased           : True


model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



STEP 2 — Model loaded
  Model class  : BertForTokenClassification
  Num labels   : 9
  id2label     : {0: 'O', 1: 'B-MISC', 2: 'I-MISC', 3: 'B-PER', 4: 'I-PER', 5: 'B-ORG', 6: 'I-ORG', 7: 'B-LOC', 8: 'I-LOC'}
  label2id     : {'B-LOC': 7, 'B-MISC': 1, 'B-ORG': 5, 'B-PER': 3, 'I-LOC': 8, 'I-MISC': 2, 'I-ORG': 6, 'I-PER': 4, 'O': 0}

STEP 3 — Input tokenized
  Raw text       : 'My name is Sarah and I live in London.'
  input_ids      : tensor([[ 101, 1422, 1271, 1110, 3696, 1105,  146, 1686, 1107, 1498,  119,  102]])
  attention_mask : tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
  Tokens         : ['[CLS]', 'My', 'name', 'is', 'Sarah', 'and', 'I', 'live', 'in', 'London', '.', '[SEP]']
  Num tokens     : 12

STEP 4 — Forward pass complete
  Output type  : TokenClassifierOutput

STEP 5 — Post-processing
  Logits shape  : torch.Size([1, 12, 9])
  Pred id shape : torch.Size([12])

  Token — label predictions:
  Token           Label      Confidence
  ─────────────── ────────── ──────────

In [7]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch
import torch.nn.functional as F

# ── AUTOMODEL FOR QUESTION ANSWERING (deepset/roberta-base-squad2) ────────────
# Task         : Extractive QA — find the answer span inside a given context
#                does NOT generate answers — only points to where the answer is
# Architecture : RoBERTa-base — 12 transformer layers, 12 heads, hidden 768
#                RoBERTa = BERT retrained longer, larger batches, no NSP task
#                no token_type_ids — RoBERTa dropped segment embeddings
# Head         : two separate Linear(768 → 1) layers — one for start, one for end
#                each produces one scalar score per token position
#                argmax over seq_len axis → predicted start and end index
# Fine-tuned on: SQuAD 2.0 — includes unanswerable questions
#                model learns to output [CLS] as start when no answer exists
# Output       : start_logits [B, seq_len] + end_logits [B, seq_len]
# Use for      : reading comprehension, document QA, FAQ extraction
# ─────────────────────────────────────────────────────────────────────────────

# ── STEP 1 : Load tokenizer ───────────────────────────────────────────────────
# RoBERTa uses BPE (Byte-Pair Encoding) — different from BERT's WordPiece
# BPE — merges frequent byte pairs iteratively to build vocab
#   BERT WordPiece : "playing" → ["play", "##ing"]   (## marks continuation)
#   RoBERTa BPE    : "playing" → ["play", "ing"]     (no ## marker, uses Ġ for spaces)
# Vocab size 50,265 — larger than BERT's 30,522
# Cased by default — RoBERTa preserves capitalisation
tokenizer = AutoTokenizer.from_pretrained("deepset/roberta-base-squad2")

print("─" * 60)
print("STEP 1 — Tokenizer loaded")
print(f"  Tokenizer class : {type(tokenizer).__name__}")
print(f"  Vocab size      : {tokenizer.vocab_size:,}")
print(f"  [CLS] token     : {tokenizer.cls_token!r}  id — {tokenizer.cls_token_id}")
print(f"  [SEP] token     : {tokenizer.sep_token!r}  id — {tokenizer.sep_token_id}")
#   RoBERTa uses <s> and </s> instead of [CLS] and [SEP]


# ── STEP 2 : Load model ───────────────────────────────────────────────────────
# AutoModelForQuestionAnswering — transformer body + two linear heads
# start head — Linear(768 → 1) per token → start_logits [B, seq_len]
# end head   — Linear(768 → 1) per token → end_logits   [B, seq_len]
# both heads are independent — end position is NOT conditioned on start
# model.eval() — disables dropout for deterministic inference
model = AutoModelForQuestionAnswering.from_pretrained("deepset/roberta-base-squad2")
model.eval()

print("\nSTEP 2 — Model loaded")
print(f"  Model class  : {type(model).__name__}")
print(f"  Num layers   : {model.config.num_hidden_layers}")
print(f"  Hidden size  : {model.config.hidden_size}")
print(f"  Num heads    : {model.config.num_attention_heads}")


# ── STEP 3 : Tokenize question + context as a pair ────────────────────────────
# QA requires two inputs: question and context (passage to search for answer)
# tokenizer(question, context) — sentence-pair encoding
#   RoBERTa format: <s> question </s> </s> context </s>
#   BERT format   : [CLS] question [SEP] context [SEP]
# model learns to read the question first, then scan context for the answer span
# question is always the first argument — this is convention, must not be swapped
question = "Where does Sarah live?"
context  = "My name is Sarah and I live in London, England."
inputs   = tokenizer(question, context, return_tensors="pt")

print("\nSTEP 3 — Input tokenized (question + context pair)")
print(f"  Question       : {question!r}")
print(f"  Context        : {context!r}")
print(f"  input_ids      : {inputs['input_ids']}")
print(f"  attention_mask : {inputs['attention_mask']}")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(f"  Tokens         : {tokens}")
print(f"  Num tokens     : {len(tokens)}")
#   ['<s>', 'Where', 'Ġdoes', 'ĠSarah', 'Ġlive', '?', '</s>', '</s>',
#    'ĠMy', 'Ġname', 'Ġis', 'ĠSarah', 'Ġand', 'ĠI', 'Ġlive', 'Ġin',
#    'ĠLondon', ',', 'ĠEngland', '.', '</s>']
#   Ġ prefix — RoBERTa BPE marker meaning "preceded by a space"
#   double </s></s> — RoBERTa's sentence boundary marker (BERT uses single [SEP])


# ── STEP 4 : Forward pass ─────────────────────────────────────────────────────
# all tokens attend to all other tokens across question AND context
# this lets context tokens "see" what the question is asking
# after 12 layers: hidden state shape [1, seq_len, 768] — one vector per token
# start head: Linear(768 → 1) at every position → start_logits [1, seq_len]
# end head  : Linear(768 → 1) at every position → end_logits   [1, seq_len]
with torch.no_grad():
    outputs = model(**inputs)

print("\nSTEP 4 — Forward pass complete")
print(f"  Output type      : {type(outputs).__name__}")
#   QuestionAnsweringModelOutput
print(f"  start_logits shape : {outputs.start_logits.shape}")
print(f"  end_logits shape   : {outputs.end_logits.shape}")
#   both — torch.Size([1, 21])  — one score per token position


# ── STEP 5 : Extract answer span ─────────────────────────────────────────────
# start_logits — score for each token being the answer start position
# end_logits   — score for each token being the answer end position
# argmax independently on each → most likely start and end index
# NOTE — no constraint that end >= start here; for robust decoding see Step 7
start_logits = outputs.start_logits
end_logits   = outputs.end_logits

start_pos = start_logits.argmax(dim=-1).item()
end_pos   = end_logits.argmax(dim=-1).item()

# slice the token IDs from start to end (inclusive) → decode to string
all_tokens = inputs["input_ids"][0]
answer_ids = all_tokens[start_pos : end_pos + 1]
answer     = tokenizer.decode(answer_ids, skip_special_tokens=True)

print("\nSTEP 5 — Answer span extracted")
print(f"  start_pos      : {start_pos}  → token {tokens[start_pos]!r}")
print(f"  end_pos        : {end_pos}    → token {tokens[end_pos]!r}")
print(f"  answer_ids     : {answer_ids.tolist()}")
print(f"  Answer         : {answer!r}")
#   Answer : 'London, England'


# ── STEP 6 : Confidence scores ────────────────────────────────────────────────
# softmax over seq_len axis → probability that each position is start/end
# NOT the same as class probabilities (no competing labels here)
# these probabilities say: "of all token positions, how likely is each one?"
start_probs = F.softmax(start_logits, dim=-1)[0]
end_probs   = F.softmax(end_logits,   dim=-1)[0]

# answer confidence — product of start and end probabilities at chosen positions
answer_score = (start_probs[start_pos] * end_probs[end_pos]).item()

print("\nSTEP 6 — Confidence scores")
print(f"  Start prob at pos {start_pos:>2}  : {start_probs[start_pos].item():.4f}")
print(f"  End prob at pos   {end_pos:>2}  : {end_probs[end_pos].item():.4f}")
print(f"  Answer score       : {answer_score:.4f}")
#   product of start × end prob — rough overall confidence

print("\n  Top 5 start positions:")
top_start = start_probs.topk(5)
for score, idx in zip(top_start.values, top_start.indices):
    print(f"    pos {idx.item():>2} — {tokens[idx.item()]:15} — {score.item():.4f}")

print("\n  Top 5 end positions:")
top_end = end_probs.topk(5)
for score, idx in zip(top_end.values, top_end.indices):
    print(f"    pos {idx.item():>2} — {tokens[idx.item()]:15} — {score.item():.4f}")


# ── STEP 7 : Unanswerable questions (SQuAD 2.0) ──────────────────────────────
# SQuAD 2.0 — half of questions have no answer in the context
# model trained to output <s> / [CLS] (position 0) when no answer exists
# position 0 = <s> token — never a valid answer token
# so: if argmax lands on position 0 → model says "no answer found"
#
# null score — logit score when model predicts no answer:
#   null_score = start_logits[0][0] + end_logits[0][0]   (both point to [CLS])
# answer score — logit score for the best valid span:
#   answer_score = start_logits[0][start_pos] + end_logits[0][end_pos]
# decision: answer_score > null_score → answer exists, else no answer
null_score   = (start_logits[0][0] + end_logits[0][0]).item()
span_score   = (start_logits[0][start_pos] + end_logits[0][end_pos]).item()

print("\nSTEP 7 — Unanswerable check (SQuAD 2.0 style)")
print(f"  Null score (pos 0 + pos 0) : {null_score:.4f}")
print(f"  Span score (start + end)   : {span_score:.4f}")
if span_score > null_score:
    print(f"  Decision — answer found    : {answer!r}")
else:
    print(f"  Decision — no answer found in context")
#   span_score > null_score → answer exists → "London, England"


# ── SEQUENCE vs TOKEN vs QA — OUTPUT SHAPE COMPARISON ────────────────────────
# AutoModelForSequenceClassification
#   logits — [B, num_labels]            one vector per sentence
#   argmax over dim=-1 → predicted class
#
# AutoModelForTokenClassification
#   logits — [B, seq_len, num_labels]   one vector per token
#   argmax over dim=-1 → predicted class per token
#
# AutoModelForQuestionAnswering
#   start_logits — [B, seq_len]         one scalar per token
#   end_logits   — [B, seq_len]         one scalar per token
#   argmax over dim=-1 → start index, end index → slice → decode
# ─────────────────────────────────────────────────────────────────────────────


# ── DRY RUN : question="Where does Sarah live?" context="My name is Sarah..." ─

# Stage 1 — Tokenize as sentence pair
# RoBERTa format: <s> question </s></s> context </s>
# token       pos   note
# <s>          0    sentence start (= [CLS] in BERT)
# Where        1    question token
# Ġdoes        2    Ġ = space before "does" in BPE
# ĠSarah       3    question token
# Ġlive        4    question token
# ?            5    punctuation
# </s>         6    sentence separator
# </s>         7    RoBERTa double separator between segments
# ĠMy          8    context starts here
# Ġname        9
# Ġis         10
# ĠSarah      11
# Ġand        12
# ĠI          13
# Ġlive       14
# Ġin         15
# ĠLondon     16   ← answer start
# ,           17
# ĠEngland    18   ← answer end
# .           19
# </s>        20   context end

# Stage 2 — Forward pass
# cross-attention between question and context:
#   "live" in question attends strongly to "live" in context (pos 14)
#   "Sarah" in question attends to "Sarah" in context (pos 11)
# start head scores — peak at pos 16 (ĠLondon)
# end head scores   — peak at pos 18 (ĠEngland)

# Stage 3 — Span extraction
# start_pos = 16 → ĠLondon
# end_pos   = 18 → ĠEngland
# answer_ids = input_ids[16:19]
# tokenizer.decode(answer_ids) → "London, England"

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

────────────────────────────────────────────────────────────
STEP 1 — Tokenizer loaded
  Tokenizer class : RobertaTokenizer
  Vocab size      : 50,265
  [CLS] token     : '<s>'  id — 0
  [SEP] token     : '</s>'  id — 2


model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



STEP 2 — Model loaded
  Model class  : RobertaForQuestionAnswering
  Num layers   : 12
  Hidden size  : 768
  Num heads    : 12

STEP 3 — Input tokenized (question + context pair)
  Question       : 'Where does Sarah live?'
  Context        : 'My name is Sarah and I live in London, England.'
  input_ids      : tensor([[    0, 13841,   473,  4143,   697,   116,     2,     2,  2387,   766,
            16,  4143,     8,    38,   697,    11,   928,     6,  1156,     4,
             2]])
  attention_mask : tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
  Tokens         : ['<s>', 'Where', 'Ġdoes', 'ĠSarah', 'Ġlive', '?', '</s>', '</s>', 'My', 'Ġname', 'Ġis', 'ĠSarah', 'Ġand', 'ĠI', 'Ġlive', 'Ġin', 'ĠLondon', ',', 'ĠEngland', '.', '</s>']
  Num tokens     : 21

STEP 4 — Forward pass complete
  Output type      : QuestionAnsweringModelOutput
  start_logits shape : torch.Size([1, 21])
  end_logits shape   : torch.Size([1, 21])

STEP 5 — Answer span extracted
  start_

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F

# ── AUTOMODEL FOR CAUSAL LM (gpt2) ───────────────────────────────────────────
# Task         : Natural Language Generation — next-token prediction
# Architecture : GPT-2 — decoder-only transformer (no encoder)
#                12 transformer layers, causal (left-to-right) self-attention
#                CANNOT look at future tokens — each token only attends to past
# Head         : Linear layer applied to EVERY token's hidden state → vocab_size logits
#                NOT just [CLS] — every position predicts the next token
# Pre-trained  : WebText corpus (~40GB of Reddit outbound links)
# Output       : logits per token position → softmax → sample/greedy → token id → decode
# Key insight  : during generation, model runs repeatedly — each step adds one new token
# ─────────────────────────────────────────────────────────────────────────────────────────────


# ── STEP 1 : Load tokenizer ───────────────────────────────────────────────────
# GPT-2 uses Byte-Pair Encoding (BPE) — NOT WordPiece like BERT
# BPE starts with characters, merges the most frequent pairs iteratively
# Result: subword tokens — common words → single token, rare words → multiple tokens
# GPT-2 has NO [CLS] or [SEP] tokens — it is decoder-only, no classification head design
# GPT-2 tokenizer has NO pad token by default — important for batching (set manually if needed)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

print("—" * 60)
print("STEP 1 — Tokenizer loaded")
print(f"  Tokenizer class : {type(tokenizer).__name__}")
print(f"  Vocab size      : {tokenizer.vocab_size:,}")          # 50,257
print(f"  EOS token       : {tokenizer.eos_token!r}")           # '<|endoftext|>'
print(f"  EOS token id    : {tokenizer.eos_token_id}")          # 50256
print(f"  PAD token       : {tokenizer.pad_token!r}")           # None — GPT-2 has no pad token
# BPE vs WordPiece:
#   WordPiece (BERT) : splits on morphology — "playing" → ["play", "##ing"]
#   BPE (GPT-2)      : splits on raw byte frequencies — " fantastic" is ONE token (space included)
#                      leading space is part of the token — "Ġfantastic" internally


# ── STEP 2 : Load model ───────────────────────────────────────────────────────
# AutoModelForCausalLM adds an LM head — a single Linear(hidden_size → vocab_size)
# applied to EVERY token position in the sequence, not just the last one
# This is because during training, ALL positions predict the next token simultaneously
# (teacher forcing — ground truth is shifted right by 1)
# During inference (generation): only the LAST position's logits matter
# model.eval() → disables dropout, ensures deterministic output
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()

print("\nSTEP 2 — Model loaded")
print(f"  Model class     : {type(model).__name__}")
print(f"  Vocab size      : {model.config.vocab_size:,}")       # 50,257
print(f"  Hidden size     : {model.config.n_embd}")             # 768
print(f"  Num layers      : {model.config.n_layer}")            # 12
print(f"  Num attn heads  : {model.config.n_head}")             # 12
print(f"  Max context len : {model.config.n_positions}")        # 1024 tokens


# ── STEP 3 : Tokenize input ───────────────────────────────────────────────────
# return_tensors="pt" → PyTorch tensors
# GPT-2 tokenizer encodes the EXACT string — no lowercasing (BPE is case-sensitive)
# Leading spaces matter: " once" and "once" are DIFFERENT tokens in BPE
# No [CLS]/[SEP] added — GPT-2 has none
text = "Once upon a time"
inputs = tokenizer(text, return_tensors="pt")

print("\nSTEP 3 — Input tokenized")
print(f"  Raw text       : {text!r}")
print(f"  input_ids      : {inputs['input_ids']}")
print(f"  attention_mask : {inputs['attention_mask']}")
print(f"  Tokens         : {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].tolist())}")
#   ['Once', 'Ġupon', 'Ġa', 'Ġtime']
#   Ġ = BPE symbol for a leading space — it is PART of the token, not a separate piece


# ── STEP 4 : Forward pass (logits only, no generation) ───────────────────────
# torch.no_grad() → no gradient tracking → saves memory, speeds up inference
# model(**inputs) returns CausalLMOutputWithCrossAttentions
# .logits shape: [batch_size, seq_len, vocab_size]
#   — every input token position produces a distribution over all 50,257 vocab tokens
#   — position i predicts what token should come at position i+1
#   — during training: logits[i] is compared against label[i+1] to compute cross-entropy loss
#   — during inference: logits[-1] (last token) is sampled/greedy-decoded for the next token
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

print("\nSTEP 4 — Forward pass complete")
print(f"  Output type    : {type(outputs).__name__}")
print(f"  Logits shape   : {logits.shape}")
#   torch.Size([1, 4, 50257])
#   batch=1, seq_len=4 (Once / upon / a / time), vocab_size=50257

# Next-token prediction from the LAST position
next_token_logits = logits[0, -1, :]          # shape: [50257] — logits for token after "time"
next_token_probs  = F.softmax(next_token_logits, dim=-1)
top5              = next_token_probs.topk(5)   # top 5 predicted continuations

print("\n  Top-5 next token predictions after 'Once upon a time':")
for prob, idx in zip(top5.values, top5.indices):
    token = tokenizer.decode([idx.item()])
    print(f"    {token!r:20s}  {prob.item():.4f}")
#   ' there'          0.1023
#   ','               0.0912
#   ' when'           0.0571
#   ' a'              0.0488
#   ' in'             0.0341


# ── STEP 5 : Generation — model.generate() ───────────────────────────────────
# model.generate() is an autoregressive loop built into HuggingFace
# Internally it runs:
#   while not done:
#       logits = model(current_input_ids)       # forward pass
#       next_id = sample_or_greedy(logits[:,-1,:])  # pick next token
#       current_input_ids = concat([current_input_ids, next_id])  # append
#       if next_id == eos_token_id or len >= max: break
#
# max_new_tokens : how many NEW tokens to generate (not counting the prompt)
# do_sample      : True → sample from distribution, False → greedy (always pick argmax)
# temperature    : scales logits BEFORE softmax
#                  T < 1.0 → sharper distribution → more repetitive, predictable
#                  T > 1.0 → flatter distribution → more random, creative, risky
#                  T = 1.0 → unchanged distribution (default)
# top_k          : only consider top-k tokens at each step → cuts off the long tail
# top_p (nucleus): only consider smallest set of tokens whose cumulative prob ≥ p
#                  adapts dynamically — uses fewer tokens when model is very confident

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8,
        top_k=50,
    )

print("\nSTEP 5 — Generation complete")
print(f"  output_ids shape : {output_ids.shape}")
#   torch.Size([1, 54])  — 4 prompt tokens + 50 new tokens

# output_ids includes the ORIGINAL prompt tokens too — slice if you want only new text
# skip_special_tokens=True removes <|endoftext|> from the output string
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(f"  Generated text   : {generated_text!r}")

# To decode ONLY the newly generated part (excluding the prompt):
new_ids_only   = output_ids[0, inputs["input_ids"].shape[1]:]
new_text_only  = tokenizer.decode(new_ids_only, skip_special_tokens=True)
print(f"  New tokens only  : {new_text_only!r}")


# ── STEP 6 : Training mode — forward pass with loss ──────────────────────────
# When you pass labels=, the model computes cross-entropy loss internally
# labels are the input_ids SHIFTED LEFT by 1:
#   input : [Once]  [upon]  [a]    [time]
#   label : [upon]  [a]    [time]  [EOS]   ← what each position should predict
# By convention in HuggingFace: pass labels = input_ids (same tensor)
# The model handles the shift internally — it does NOT predict from position 0
# Any position where label == -100 is IGNORED in the loss (masking trick)

model.train()   # switch back to training mode for this demonstration

labels  = inputs["input_ids"].clone()     # shape: [1, 4]
outputs_with_loss = model(**inputs, labels=labels)

print("\nSTEP 6 — Training forward pass (with loss)")
print(f"  Loss   : {outputs_with_loss.loss.item():.4f}")    # cross-entropy loss (scalar)
print(f"  Logits : {outputs_with_loss.logits.shape}")       # [1, 4, 50257] same as before
#   lower loss = model is more confident about the correct next tokens in this sequence

model.eval()    # always switch back after demo


# ── GENERATION STRATEGY COMPARISON ───────────────────────────────────────────
# Strategy          do_sample   Key params        Behaviour
# ─────────────────────────────────────────────────────────────────────────────
# Greedy            False       —                 Always argmax — fast, repetitive
# Beam search       False       num_beams=5       Explores top-N sequences in parallel
# Temperature       True        temperature=0.8   Adjusts sharpness of distribution
# Top-K             True        top_k=50          Sample only from top 50 tokens
# Top-P (nucleus)   True        top_p=0.9         Sample from tokens summing to 90% prob
# Combined          True        temp + top_k/p    Most common in practice (GPT-style)
# ─────────────────────────────────────────────────────────────────────────────
# Production default: do_sample=True, temperature=0.7~1.0, top_p=0.9


# ── AUTOMODEL VARIANTS — TASK REFERENCE ──────────────────────────────────────
# Class                                Head added on top of base model
# ─────────────────────────────────────────────────────────────────────────────
# AutoModel                            None — raw hidden states only
# AutoModelForSequenceClassification   Linear([CLS] → num_labels)
# AutoModelForTokenClassification      Linear(each token → num_labels) — NER, POS
# AutoModelForQuestionAnswering        Two linears — start logits + end logits per token
# AutoModelForMaskedLM                 Linear(each token → vocab_size) — fill [MASK]
# AutoModelForCausalLM                 Linear(each token → vocab_size) — next token  ← THIS FILE
# AutoModelForSeq2SeqLM                Encoder + Decoder heads — translation, summarise
# AutoModelForMultipleChoice           Linear([CLS] per choice → 1 score) — MCQ
# ─────────────────────────────────────────────────────────────────────────────


# ── OUTPUT DATACLASS FIELDS ───────────────────────────────────────────────────
# CausalLMOutputWithCrossAttentions
#   .loss              → scalar (only present when labels= is passed)
#   .logits            → shape [B, seq_len, vocab_size]
#   .past_key_values   → KV cache tuple — reuse on the next generation step (speed)
#   .hidden_states     → tuple per layer (needs output_hidden_states=True)
#   .attentions        → attention weight matrices (needs output_attentions=True)
#
# KV cache (.past_key_values):
#   Each generation step recomputes keys+values for ALL tokens by default
#   KV cache stores previously computed keys+values → only new token is computed
#   Huge speedup for long sequences — model.generate() uses it automatically


# ── DRY RUN : text = "Once upon a time" ──────────────────────────────────────

# Stage 1 — BPE Tokenization (case-sensitive, space-aware)
# "Once upon a time"
# GPT-2 BPE encodes space as part of the following token (Ġ prefix)
#
# word       raw bytes       BPE token       token id
# Once    →  O-n-c-e      →  Once            7454
# [space] →  merged into next token
# upon    →  Ġu-p-o-n     →  Ġupon          2402
# a       →  Ġa           →  Ġa              257
# time    →  Ġt-i-m-e     →  Ġtime          640
#
# input_ids      = tensor([[7454, 2402, 257, 640]])
# attention_mask = tensor([[   1,    1,   1,   1]])
# NOTE: no [CLS] / [SEP] — GPT-2 has neither

# Stage 2 — Embedding + Positional Encoding
# Each token id → token embedding (shape [768]) from wte (word token embedding) table
# Each position → position embedding (shape [768]) from wpe (word position embedding) table
# Final input to transformer = token_embedding + position_embedding  (element-wise add)
# GPT-2 uses learned position embeddings (not sinusoidal like original Transformer)
#
# pos  token      token_emb shape   pos_emb shape   combined shape
#  0   Once       [768]              [768]            [768]
#  1   Ġupon      [768]              [768]            [768]
#  2   Ġa         [768]              [768]            [768]
#  3   Ġtime      [768]              [768]            [768]
#
# Full sequence fed into transformer: shape [1, 4, 768]

# Stage 3 — Causal Self-Attention (per layer, 12 layers total)
# Each token attends ONLY to itself and tokens before it (causal mask)
# Attention mask pattern for seq_len=4:
#
#           Once   upon    a    time
#  Once   [  1      0      0      0  ]   ← can only see itself
#  upon   [  1      1      0      0  ]   ← sees Once + itself
#  a      [  1      1      1      0  ]   ← sees Once, upon, itself
#  time   [  1      1      1      1  ]   ← sees all previous tokens
#
# This is implemented as an additive mask: 0 → attend, -inf → block (before softmax)
# After softmax: -inf positions become 0.0 attention weight → ignored

# Stage 4 — LM Head (after 12 transformer layers)
# Each token position has a hidden state of shape [768]
# LM head = Linear(768 → 50257) applied to EVERY position
# Result: logits shape [1, 4, 50257]
#
# pos  token   hidden_state → logits            meaning
#  0   Once    [768]        → [50257]           predicts the token after "Once"
#  1   Ġupon   [768]        → [50257]           predicts the token after "Once upon"
#  2   Ġa      [768]        → [50257]           predicts the token after "Once upon a"
#  3   Ġtime   [768]        → [50257]           predicts the token after "Once upon a time" ← used
#
# Only logits[0, -1, :] matters for generation — the last position predicts what comes next

# Stage 5 — Sampling (do_sample=True, temperature=0.8, top_k=50)
# next_token_logits = logits[0, -1, :]         shape [50257]
#
# Step A — temperature scaling
#   scaled_logits = logits / 0.8
#   Effect: divide by T < 1 → logits get larger → softmax is sharper → safer choices
#
# Step B — top_k filtering
#   Keep only the top 50 logit values; set the rest to -inf
#   After softmax, the blocked tokens have probability 0
#
# Step C — softmax → probability distribution over top 50 tokens
#   probs = softmax(filtered_logits)          shape [50257] but 50,207 entries are 0
#
# Step D — sample one token from this distribution
#   torch.multinomial(probs, num_samples=1)
#   Picks an index proportional to its probability
#   Higher prob tokens are more likely but not guaranteed (unlike greedy)
#
# Step E — append sampled token id to input_ids → new input_ids shape [1, 5]
#           repeat from Stage 2 for the next step
#           continue until max_new_tokens=50 reached OR EOS token sampled

# Stage 6 — Decode output_ids back to text
# output_ids = tensor([[7454, 2402, 257, 640, ...50 new ids...]])  shape [1, 54]
# tokenizer.decode(output_ids[0], skip_special_tokens=True)
#   converts each id → its BPE token string
#   strips Ġ prefix → replaces with actual space
#   concatenates all pieces → final readable string
#
# Example final output: "Once upon a time there was a young girl who lived in a forest..."

————————————————————————————————————————————————————————————
STEP 1 — Tokenizer loaded
  Tokenizer class : GPT2Tokenizer
  Vocab size      : 50,257
  EOS token       : '<|endoftext|>'
  EOS token id    : 50256
  PAD token       : None


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


STEP 2 — Model loaded
  Model class     : GPT2LMHeadModel
  Vocab size      : 50,257
  Hidden size     : 768
  Num layers      : 12
  Num attn heads  : 12
  Max context len : 1024

STEP 3 — Input tokenized
  Raw text       : 'Once upon a time'
  input_ids      : tensor([[7454, 2402,  257,  640]])
  attention_mask : tensor([[1, 1, 1, 1]])
  Tokens         : ['Once', 'Ġupon', 'Ġa', 'Ġtime']


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



STEP 4 — Forward pass complete
  Output type    : CausalLMOutputWithCrossAttentions
  Logits shape   : torch.Size([1, 4, 50257])

  Top-5 next token predictions after 'Once upon a time':
    ','                   0.4269
    ' the'                0.0646
    ' I'                  0.0406
    ' he'                 0.0369
    ' there'              0.0294


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.



STEP 5 — Generation complete
  output_ids shape : torch.Size([1, 54])
  Generated text   : 'Once upon a time, it was a different world. In the year 2000, the world was the very thing that had been designed by those who were first to produce it.\n\nIn this world, you had to rely on the power of your imagination.\n\n'
  New tokens only  : ', it was a different world. In the year 2000, the world was the very thing that had been designed by those who were first to produce it.\n\nIn this world, you had to rely on the power of your imagination.\n\n'

STEP 6 — Training forward pass (with loss)
  Loss   : 3.7743
  Logits : torch.Size([1, 4, 50257])


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [2]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import torch.nn.functional as F

# ── AUTOMODEL FOR MASKED LM (bert-base-uncased) ───────────────────────────────
# Task         : Masked Language Modeling (MLM) — predict hidden/corrupted tokens
# Architecture : BERT — encoder-only bidirectional transformer
#                12 transformer layers, FULL self-attention (every token sees every token)
#                OPPOSITE of GPT-2 — no causal mask, context flows in BOTH directions
# Head         : Linear(hidden_size → vocab_size) + GELU + LayerNorm applied per token
#                Only the [MASK] positions matter at inference; all positions computed
# Pre-trained  : BookCorpus + English Wikipedia — 15% of tokens randomly masked
#                80% replaced with [MASK], 10% random token, 10% unchanged (training trick)
# Output       : logits per token position → softmax at [MASK] positions → predicted tokens
# Key insight  : encoder sees the FULL sequence simultaneously — left AND right context
#                "The capital of France is [MASK]" — model uses "France" + "capital" together
# ─────────────────────────────────────────────────────────────────────────────────────────────


# ── STEP 1 : Load tokenizer ───────────────────────────────────────────────────
# bert-base-uncased uses WordPiece tokenization
# "uncased" → input is lowercased before tokenization — "Paris" == "paris"
# WordPiece: starts with characters, merges common subword units
#   whole words in vocab → single token
#   rare/unseen words    → split into pieces, continuation pieces prefixed with ##
#   "unbelievable" → ["un", "##believe", "##able"]
# Special tokens:
#   [CLS] (id 101) → prepended to every sequence — used for classification tasks
#   [SEP] (id 102) → appended to mark end of sequence (or boundary between two sequences)
#   [MASK] (id 103) → the token we deliberately replace to ask model to predict
#   [PAD] (id 0)   → padding to align sequences in a batch to the same length
#   [UNK] (id 100) → any character sequence not in the vocab
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("—" * 60)
print("STEP 1 — Tokenizer loaded")
print(f"  Tokenizer class  : {type(tokenizer).__name__}")
print(f"  Vocab size       : {tokenizer.vocab_size:,}")        # 30,522
print(f"  [CLS]  id        : {tokenizer.cls_token_id}")        # 101
print(f"  [SEP]  id        : {tokenizer.sep_token_id}")        # 102
print(f"  [MASK] id        : {tokenizer.mask_token_id}")       # 103
print(f"  [PAD]  id        : {tokenizer.pad_token_id}")        # 0
# WordPiece vs BPE (GPT-2):
#   WordPiece  : splits based on likelihood maximisation — ## prefix for continuations
#   BPE        : splits based on byte-pair frequency — Ġ prefix for leading spaces
#   WordPiece is case-INsensitive here (uncased) — BPE is always case-sensitive


# ── STEP 2 : Load model ───────────────────────────────────────────────────────
# AutoModelForMaskedLM wraps the BERT encoder with an MLM head
# MLM head = Dense(hidden_size → hidden_size) + GELU + LayerNorm + Linear(hidden_size → vocab_size)
# Slightly heavier than a plain linear — the intermediate Dense+GELU helps recover masked context
# .eval() → disables dropout for deterministic inference
model = AutoModelForMaskedLM.from_pretrained("bert-base-uncased")
model.eval()

print("\nSTEP 2 — Model loaded")
print(f"  Model class      : {type(model).__name__}")
print(f"  Vocab size       : {model.config.vocab_size:,}")      # 30,522
print(f"  Hidden size      : {model.config.hidden_size}")       # 768
print(f"  Num layers       : {model.config.num_hidden_layers}") # 12
print(f"  Num attn heads   : {model.config.num_attention_heads}")# 12
print(f"  Max position emb : {model.config.max_position_embeddings}") # 512


# ── STEP 3 : Tokenize input (with [MASK] already in the string) ───────────────
# BERT's tokenizer recognises "[MASK]" as a special token and maps it to id 103
# You can also insert it programmatically: tokenizer.mask_token == "[MASK]"
# The raw string must contain the EXACT special token string — "[MASK]" not "<mask>"
# truncation=True, max_length=512 → safety for long inputs (BERT hard limit = 512)
text   = "The capital of France is [MASK]."
inputs = tokenizer(text, return_tensors="pt")

print("\nSTEP 3 — Input tokenized")
print(f"  Raw text         : {text!r}")
print(f"  input_ids        : {inputs['input_ids']}")
print(f"  attention_mask   : {inputs['attention_mask']}")
print(f"  token_type_ids   : {inputs['token_type_ids']}")       # all 0s — single sequence
print(f"  Tokens           : {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].tolist())}")
#   ['[CLS]', 'the', 'capital', 'of', 'france', 'is', '[MASK]', '.', '[SEP]']
#
# token_type_ids explanation:
#   Single sequence → all 0s
#   Two sequences (e.g. NLI: premise + hypothesis) → 0s for seq A, 1s for seq B
#   BERT was pre-trained on Next Sentence Prediction (NSP) with two sequences
#   For MLM alone (single sequence) token_type_ids are all 0 and largely irrelevant


# ── STEP 4 : Locate the [MASK] position ──────────────────────────────────────
# We need to know WHICH index in input_ids corresponds to [MASK] (id 103)
# == comparison returns a boolean tensor, .nonzero() returns indices of True positions
# as_tuple=True → returns a tuple of tensors (one per dimension) instead of Nx2 tensor
# [1] → we want the column index (position within the sequence), not the batch index [0]
mask_token_id = tokenizer.mask_token_id                            # 103
mask_pos      = (inputs["input_ids"] == mask_token_id).nonzero(as_tuple=True)[1]

print("\nSTEP 4 — [MASK] position identified")
print(f"  [MASK] token id  : {mask_token_id}")
print(f"  [MASK] position  : {mask_pos}")                # tensor([6])
print(f"  Token at pos     : {tokenizer.convert_ids_to_tokens([inputs['input_ids'][0, mask_pos].item()])}")
#   ['[MASK]']  ← confirms we found the right position


# ── STEP 5 : Forward pass ─────────────────────────────────────────────────────
# BERT processes the ENTIRE sequence in parallel — all tokens attend to all tokens
# NO autoregressive loop needed (unlike GPT-2 generation)
# Full bidirectional attention means [MASK] at position 6 can attend to:
#   "france" (position 4) → strong left context
#   "." (position 7)      → right context (only "." here, but helps with grammar)
# outputs.logits shape: [batch_size, seq_len, vocab_size]
#   = [1, 9, 30522]
#   — every token position gets a full vocabulary distribution
#   — but for MLM we ONLY care about the [MASK] position(s)
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

print("\nSTEP 5 — Forward pass complete")
print(f"  Output type      : {type(outputs).__name__}")
print(f"  Logits shape     : {logits.shape}")
#   torch.Size([1, 9, 30522])
#   batch=1, seq_len=9 ([CLS]+7 tokens+[SEP]), vocab=30,522


# ── STEP 6 : Post-process — extract predictions at [MASK] ────────────────────
# Slice out logits for ONLY the [MASK] position
# logits[0]           → batch item 0, shape [seq_len, vocab_size]
# logits[0, mask_pos] → rows corresponding to [MASK] position, shape [1, vocab_size]
# .squeeze(0)         → shape [vocab_size] — remove the leading 1 dimension
mask_logits = logits[0, mask_pos, :].squeeze(0)         # shape: [30522]

# softmax → probability distribution over entire vocabulary at the masked position
mask_probs  = F.softmax(mask_logits, dim=-1)            # shape: [30522]

# topk(5) → returns (values, indices) NamedTuple for the 5 highest probabilities
top5        = mask_probs.topk(5)

print("\nSTEP 6 — Post-processing")
print(f"  Mask logits shape : {mask_logits.shape}")      # torch.Size([30522])
print(f"  Mask probs shape  : {mask_probs.shape}")       # torch.Size([30522])
print(f"\n  Top-5 predictions at [MASK] position:")
for prob, idx in zip(top5.values, top5.indices):
    token = tokenizer.convert_ids_to_tokens([idx.item()])[0]
    print(f"    {token:15s}  {prob.item():.4f}")
#   paris           0.9123
#   lyon            0.0201
#   nice            0.0087
#   marseille       0.0055
#   toulouse        0.0031


# ── STEP 7 : Multiple [MASK] tokens ──────────────────────────────────────────
# BERT supports masking multiple positions simultaneously
# Each [MASK] position is predicted INDEPENDENTLY — not autoregressively
# "Berlin is the [MASK] of [MASK]" → predicts "capital" AND "Germany" in one forward pass
# Limitation: the predictions don't condition on each other (unlike seq2seq)
text_multi   = "Berlin is the [MASK] of [MASK]."
inputs_multi = tokenizer(text_multi, return_tensors="pt")

with torch.no_grad():
    outputs_multi = model(**inputs_multi)

logits_multi  = outputs_multi.logits
mask_positions = (inputs_multi["input_ids"] == mask_token_id).nonzero(as_tuple=True)[1]

print("\nSTEP 7 — Multiple [MASK] tokens")
print(f"  Tokens           : {tokenizer.convert_ids_to_tokens(inputs_multi['input_ids'][0].tolist())}")
print(f"  [MASK] positions : {mask_positions.tolist()}")

for pos in mask_positions:
    pos_logits = logits_multi[0, pos, :]
    pos_probs  = F.softmax(pos_logits, dim=-1)
    top1_id    = pos_probs.argmax().item()
    top1_token = tokenizer.convert_ids_to_tokens([top1_id])[0]
    print(f"  [MASK] at pos {pos.item():2d}  → {top1_token!r}  ({pos_probs[top1_id].item():.4f})")
#   [MASK] at pos  4  → 'capital'   (0.6871)
#   [MASK] at pos  6  → 'germany'   (0.8234)


# ── BERT vs GPT-2 — ARCHITECTURE COMPARISON ──────────────────────────────────
# Property               BERT (encoder-only)          GPT-2 (decoder-only)
# ─────────────────────────────────────────────────────────────────────────────
# Attention direction    Bidirectional (full)          Causal (left-to-right only)
# Special tokens         [CLS], [SEP], [MASK], [PAD]  <|endoftext|> only
# Tokenizer              WordPiece (## prefix)         BPE (Ġ prefix)
# Case sensitivity       Uncased variant exists        Always case-sensitive
# Pre-training task      MLM + NSP                     Next-token prediction
# Max sequence length    512 tokens                    1024 tokens
# Output used            Specific token positions      Last token position
# Good for               Understanding, classification Generation, completion
# ─────────────────────────────────────────────────────────────────────────────


# ── OUTPUT DATACLASS FIELDS ───────────────────────────────────────────────────
# MaskedLMOutput
#   .loss         → scalar (only present when labels= is passed for pre-training)
#   .logits       → shape [B, seq_len, vocab_size]  — main output
#   .hidden_states → tuple of tensors per layer (needs output_hidden_states=True)
#   .attentions    → attention weight matrices      (needs output_attentions=True)
#
# Passing labels= for pre-training loss:
#   labels = input_ids.clone()
#   labels[input_ids != mask_token_id] = -100   # ignore non-masked positions
#   outputs = model(**inputs, labels=labels)
#   outputs.loss  → cross-entropy only at [MASK] positions (−100 positions are skipped)


# ── DRY RUN : text = "The capital of France is [MASK]." ──────────────────────

# Stage 1 — Lowercasing (uncased model)
# "The capital of France is [MASK]."
#  → "the capital of france is [MASK]."
#  NOTE: [MASK] is a special token — it is NOT lowercased, it is preserved as-is

# Stage 2 — WordPiece tokenization (no specials yet)
# word        pieces              reason
# the      →  the                 whole word in vocab
# capital  →  capital             whole word in vocab
# of       →  of                  whole word in vocab
# france   →  france              whole word in vocab
# is       →  is                  whole word in vocab
# [MASK]   →  [MASK]              special token — no splitting
# .        →  .                   punctuation split
#
# tokens = ['the', 'capital', 'of', 'france', 'is', '[MASK]', '.']

# Stage 3 — Add special tokens + convert to ids
# pos  token       id
#  0   [CLS]      101    ← always prepended
#  1   the        1996
#  2   capital    3007
#  3   of         1997
#  4   france     2605
#  5   is         2003
#  6   [MASK]      103   ← the position we care about
#  7   .           1012
#  8   [SEP]       102   ← always appended
#
# input_ids      = tensor([[101, 1996, 3007, 1997, 2605, 2003, 103, 1012, 102]])
# attention_mask = tensor([[  1,    1,    1,    1,    1,    1,   1,    1,   1]])
# token_type_ids = tensor([[  0,    0,    0,    0,    0,    0,   0,    0,   0]])

# Stage 4 — Embedding
# Each token → token embedding [768] + position embedding [768] + token_type embedding [768]
# BERT has THREE embedding tables (unlike GPT-2 which has only two):
#   wte  : word token embeddings   shape [30522, 768]
#   wpe  : position embeddings     shape [512, 768]
#   wse  : segment (type) embeddings shape [2, 768]  ← unique to BERT (NSP pre-training)
# All three summed → layer norm → input to transformer  shape [1, 9, 768]

# Stage 5 — Bidirectional Self-Attention (12 layers)
# ALL tokens attend to ALL tokens — no causal mask
# Attention matrix for seq_len=9 is fully dense (9×9 = 81 entries, all active):
#
#          CLS  the  cap  of   fra  is  [MSK]  .   SEP
#  CLS    [ 1    1    1    1    1    1    1     1    1 ]
#  the    [ 1    1    1    1    1    1    1     1    1 ]
#  cap    [ 1    1    1    1    1    1    1     1    1 ]
#  of     [ 1    1    1    1    1    1    1     1    1 ]
#  fra    [ 1    1    1    1    1    1    1     1    1 ]
#  is     [ 1    1    1    1    1    1    1     1    1 ]
# [MSK]   [ 1    1    1    1    1    1    1     1    1 ]  ← [MASK] sees "france" directly
#  .      [ 1    1    1    1    1    1    1     1    1 ]
#  SEP    [ 1    1    1    1    1    1    1     1    1 ]
#
# [MASK] at position 6 has direct attention path to:
#   "france" at position 4 → very high attention weight → strong signal for "paris"
#   "capital" at position 2 → context that this is a geopolitical answer
#   "is" at position 5 → grammatical agreement → expects a proper noun

# Stage 6 — MLM Head applied to ALL positions
# After 12 transformer layers: hidden states shape [1, 9, 768]
# MLM head per position:
#   Dense(768 → 768) + GELU activation → shape [768]
#   LayerNorm → shape [768]
#   Linear(768 → 30522, weight tied to token embedding matrix) → shape [30522]
#
# Weight tying: the linear projection at the end SHARES weights with the token embedding table
# This reduces parameters and ensures the output space aligns with the input embedding space
#
# Logits computed for ALL 9 positions, but only position 6 ([MASK]) is used:
# logits[0, 6, :]  →  shape [30522]
#
# Top raw logits at position 6 (before softmax):
#   id 3000 ("paris")      → 8.71
#   id 5849 ("lyon")       → 4.23
#   id 3835 ("nice")       → 3.91
#   id 9419 ("marseille")  → 3.64
#   id 9140 ("toulouse")   → 3.52

# Stage 7 — Softmax → probabilities
# softmax([8.71, 4.23, 3.91, ...])
#   exp(8.71)  = 6,074.3  ← paris dominates
#   exp(4.23)  =   68.7
#   exp(3.91)  =   49.9
#   exp(3.64)  =   38.1
#   exp(3.52)  =   33.8   (plus ~30,517 more tiny values)
#   sum        ≈ 6,661.0
#
# probabilities:
#   paris       6074.3 / 6661.0 = 0.9119  → 91.2%
#   lyon          68.7 / 6661.0 = 0.0103  →  1.0%
#   nice          49.9 / 6661.0 = 0.0075  →  0.7%
#   marseille     38.1 / 6661.0 = 0.0057  →  0.6%
#   toulouse      33.8 / 6661.0 = 0.0051  →  0.5%
#
# argmax = id 3000 → tokenizer.convert_ids_to_tokens([3000]) → "paris"
# final answer: PARIS (91.2% confidence)
# BERT answers correctly because "france" + "capital" together in bidirectional context
# is unambiguous — a GPT-style left-to-right model would struggle if "france" came AFTER [MASK]

————————————————————————————————————————————————————————————
STEP 1 — Tokenizer loaded
  Tokenizer class  : BertTokenizer
  Vocab size       : 30,522
  [CLS]  id        : 101
  [SEP]  id        : 102
  [MASK] id        : 103
  [PAD]  id        : 0


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



STEP 2 — Model loaded
  Model class      : BertForMaskedLM
  Vocab size       : 30,522
  Hidden size      : 768
  Num layers       : 12
  Num attn heads   : 12
  Max position emb : 512

STEP 3 — Input tokenized
  Raw text         : 'The capital of France is [MASK].'
  input_ids        : tensor([[ 101, 1996, 3007, 1997, 2605, 2003,  103, 1012,  102]])
  attention_mask   : tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])
  token_type_ids   : tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]])
  Tokens           : ['[CLS]', 'the', 'capital', 'of', 'france', 'is', '[MASK]', '.', '[SEP]']

STEP 4 — [MASK] position identified
  [MASK] token id  : 103
  [MASK] position  : tensor([6])
  Token at pos     : ['[MASK]']

STEP 5 — Forward pass complete
  Output type      : MaskedLMOutput
  Logits shape     : torch.Size([1, 9, 30522])

STEP 6 — Post-processing
  Mask logits shape : torch.Size([30522])
  Mask probs shape  : torch.Size([30522])

  Top-5 predictions at [MASK] position:
    paris            0.4168
    lille   

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import torch.nn.functional as F

# ── AUTOMODEL FOR SEQ2SEQ LM (facebook/bart-large-cnn) ───────────────────────
# Task         : Sequence-to-sequence generation — input sequence → output sequence
# Architecture : BART — encoder + decoder transformer (full encoder-decoder stack)
#                Encoder : bidirectional (like BERT) — reads and understands the input
#                Decoder : causal / autoregressive (like GPT) — generates output left-to-right
#                Encoder : 12 layers, Decoder : 12 layers — 400M parameters total
# Head         : Linear(hidden_size → vocab_size) on EVERY decoder output position
#                Same as CausalLM head but applied to decoder hidden states, not encoder
# Fine-tuned on: CNN / DailyMail news summarisation dataset
# Pre-training : BART-specific — encoder input is CORRUPTED text (masked spans, shuffled)
#                decoder reconstructs the ORIGINAL clean text — learns to denoise
# Output       : beam search or sampling over decoder vocab at each step → token ids → decode
# Key insight  : encoder runs ONCE on the full input → produces rich context vectors
#                decoder runs AUTOREGRESSIVELY — at each step it attends to encoder output
#                via cross-attention AND to its own previously generated tokens via self-attention
# ─────────────────────────────────────────────────────────────────────────────────────────────


# ── STEP 1 : Load tokenizer ───────────────────────────────────────────────────
# BART uses BPE (same family as GPT-2/RoBERTa) — NOT WordPiece like BERT
# "large" variant → same vocab as RoBERTa → 50,265 tokens
# Special tokens:
#   <s>   (id 0)  → BOS — Beginning Of Sequence, prepended to decoder input
#   </s>  (id 2)  → EOS — End Of Sequence, appended to encoder input AND generation stop signal
#   <pad> (id 1)  → padding for batched inputs
#   <mask>(id 50264) → used only during BART pre-training (denoising), not at inference
# Unlike GPT-2, BART DOES have a pad token — needed because encoder processes full input at once
# Unlike BERT, no [CLS] / [SEP] / [MASK] — BART uses <s> and </s> instead
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")

print("—" * 60)
print("STEP 1 — Tokenizer loaded")
print(f"  Tokenizer class  : {type(tokenizer).__name__}")
print(f"  Vocab size       : {tokenizer.vocab_size:,}")          # 50,265
print(f"  BOS token        : {tokenizer.bos_token!r}")           # '<s>'
print(f"  EOS token        : {tokenizer.eos_token!r}")           # '</s>'
print(f"  PAD token        : {tokenizer.pad_token!r}")           # '<pad>'
print(f"  BOS id           : {tokenizer.bos_token_id}")          # 0
print(f"  EOS id           : {tokenizer.eos_token_id}")          # 2
print(f"  PAD id           : {tokenizer.pad_token_id}")          # 1
# BPE (BART/GPT-2/RoBERTa) vs WordPiece (BERT):
#   BPE      : byte-level, case-sensitive, Ġ prefix for space-prefixed tokens
#   WordPiece: character-level, uncased variant exists, ## prefix for continuations


# ── STEP 2 : Load model ───────────────────────────────────────────────────────
# AutoModelForSeq2SeqLM loads BOTH encoder and decoder
# Encoder → BidirectionalTransformer (all tokens attend to all tokens)
# Decoder → CausalTransformer with TWO attention mechanisms per layer:
#   1. Masked self-attention    : decoder attends to its own past output tokens (causal)
#   2. Cross-attention          : decoder attends to ALL encoder hidden states
#      cross-attention is what makes seq2seq powerful — decoder "looks back" at encoder
# LM head : Linear(1024 → 50265) applied to each decoder output position
#            weight-tied to decoder token embeddings (same as BERT MLM head tying)
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")
model.eval()

print("\nSTEP 2 — Model loaded")
print(f"  Model class      : {type(model).__name__}")
print(f"  Encoder layers   : {model.config.encoder_layers}")     # 12
print(f"  Decoder layers   : {model.config.decoder_layers}")     # 12
print(f"  Hidden size      : {model.config.d_model}")            # 1024
print(f"  Attn heads       : {model.config.encoder_attention_heads}") # 16
print(f"  Max position     : {model.config.max_position_embeddings}") # 1024


# ── STEP 3 : Tokenize the input (encoder side) ───────────────────────────────
# truncation=True + max_length=1024 → hard cap at BART's context window
# BART tokenizer automatically adds <s> at the start and </s> at the end
# This is the ENCODER input — the text you want to summarise/translate/correct
# max_length refers to the INPUT (source) — output length is controlled in .generate()
text   = (
    "The Amazon rainforest is the world's largest tropical rainforest, covering over "
    "5.5 million square kilometres. It represents over half of the planet's remaining "
    "rainforests and comprises the largest and most biodiverse tract of tropical rainforest "
    "in the world, with an estimated 390 billion individual trees divided into 16,000 species. "
    "More than 30 million people of 350 different ethnic groups live in the Amazon, "
    "which are subdivided into 9 different national political systems and 3,344 formally "
    "acknowledged indigenous territories. The Amazon represents 10 percent of all species "
    "on Earth and has been described as the lungs of the planet for its role in regulating "
    "the global climate by absorbing carbon dioxide and producing oxygen."
)
inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)

print("\nSTEP 3 — Input tokenized (encoder side)")
print(f"  input_ids shape  : {inputs['input_ids'].shape}")
print(f"  attention_mask   : {inputs['attention_mask'].shape}")
print(f"  First 10 tokens  : {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0][:10].tolist())}")
#   ['<s>', 'The', 'ĠAmazon', 'Ġrainforest', 'Ġis', 'Ġthe', 'Ġworld', "'s", 'Ġlargest', 'Ġtropical']
print(f"  Last  3 tokens   : {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0][-3:].tolist())}")
#   ['Ġoxygen', '.', '</s>']   ← EOS always appended by BART tokenizer


# ── STEP 4 : Forward pass (manual encoder + decoder) ─────────────────────────
# model.generate() handles this automatically, but breaking it apart shows the internals:
#
# ENCODER: run once on the full input → produces encoder_hidden_states
#   shape: [batch, src_seq_len, hidden_size]  = [1, N, 1024]
#   these are the rich contextual representations of the source text
#   decoder will attend to these at every generation step via cross-attention
#
# DECODER: runs autoregressively — one step at a time
#   At step 0: decoder_input_ids = [<s>]  (forced BOS start token)
#   At step 1: decoder_input_ids = [<s>, first_generated_token]
#   At step t: decoder_input_ids = [<s>, tok1, tok2, ..., tok_t]
#   Each step: masked self-attn over own past tokens + cross-attn over encoder output → logits
#
# Here we do ONE manual decoder step to inspect the internals:
with torch.no_grad():
    encoder_outputs = model.model.encoder(**inputs)          # encode the full input once

encoder_hidden_states = encoder_outputs.last_hidden_state   # shape: [1, src_len, 1024]

# Decoder step 0: start with just the BOS token <s>
decoder_input_ids = torch.tensor([[tokenizer.bos_token_id]])  # shape: [1, 1]

with torch.no_grad():
    decoder_outputs = model.model.decoder(
        input_ids             = decoder_input_ids,
        encoder_hidden_states = encoder_hidden_states,
        encoder_attention_mask= inputs["attention_mask"],
    )

# LM head: project decoder hidden state → vocab logits
step0_logits = model.lm_head(decoder_outputs.last_hidden_state)  # shape: [1, 1, 50265]

print("\nSTEP 4 — Manual encoder + decoder step")
print(f"  Encoder hidden states shape : {encoder_hidden_states.shape}")
#   torch.Size([1, src_len, 1024])
print(f"  Decoder output shape        : {decoder_outputs.last_hidden_state.shape}")
#   torch.Size([1, 1, 1024])
print(f"  Step-0 logits shape         : {step0_logits.shape}")
#   torch.Size([1, 1, 50265])

# What does the model predict as the FIRST summary token?
first_token_probs = F.softmax(step0_logits[0, 0, :], dim=-1)
top1_id           = first_token_probs.argmax().item()
print(f"  Predicted first token       : {tokenizer.decode([top1_id])!r}")
#   'The'  ← model starts the summary with "The" — makes sense for a CNN/DailyMail style


# ── STEP 5 : Generation — model.generate() with beam search ──────────────────
# model.generate() for seq2seq internally:
#   1. Run encoder ONCE on inputs → encoder_hidden_states
#   2. Initialise num_beams=4 decoder sequences, each starting with [<s>]
#   3. At each step, expand each beam: top-K continuations per beam
#   4. Keep the global top num_beams sequences by cumulative log-probability
#   5. Stop when all beams have generated </s> OR max_new_tokens reached
#
# Beam search vs greedy vs sampling (recap):
#   greedy     : always pick argmax — fast, tends to be repetitive
#   beam search: explore top-N sequences in parallel — better for fixed-length tasks
#   sampling   : stochastic — better for open-ended generation (stories, chat)
#   For summarisation: beam search is standard — you want the most probable coherent summary
#
# Key generation parameters:
#   max_new_tokens      : max tokens the DECODER generates (not counting encoder input)
#   min_length          : force at least this many tokens before </s> is allowed
#                         prevents degenerate one-word summaries
#   num_beams           : beam width — higher = better quality, slower
#   early_stopping      : stop all beams as soon as num_beams complete sequences found
#   no_repeat_ngram_size: block any n-gram that has already appeared → reduces repetition
#                         3 means no 3-word phrase appears twice in the output
#   length_penalty      : > 1.0 rewards longer sequences, < 1.0 penalises them
#                         default 1.0 for BART-CNN (tuned for news summaries)
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens       = 150,
        min_length           = 50,
        num_beams            = 4,
        early_stopping       = True,
        no_repeat_ngram_size = 3,
    )

print("\nSTEP 5 — Beam search generation complete")
print(f"  output_ids shape  : {output_ids.shape}")
#   torch.Size([1, M])  — M = number of generated tokens (includes BOS/EOS)
print(f"  First 5 token ids : {output_ids[0, :5].tolist()}")
#   [0, 133, 3660, 35567, 16]   ← [<s>, 'The', 'ĠAmazon', ...]
print(f"  Last  3 token ids : {output_ids[0, -3:].tolist()}")
#   [..., 479, 2]               ← ['Ġoxygen', '.', '</s>']

# tokenizer.decode():
#   converts each id → its BPE string piece
#   skip_special_tokens=True → removes <s> and </s> from the output
#   strips Ġ → real spaces, assembles the final readable string
summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(f"\n  Summary:\n  {summary}")
#   "The Amazon rainforest covers over 5.5 million square kilometres and is home
#    to more than 30 million people. It has been described as the lungs of the
#    planet for its role in regulating the global climate."


# ── STEP 6 : T5 / Flan-T5 — task-prefix style ────────────────────────────────
# T5 treats EVERY NLP task as text-to-text:
#   summarisation : "summarize: <document>"
#   translation   : "translate English to French: <text>"
#   classification: "sst2 sentence: <text>"   → outputs "positive" / "negative"
#   grammar       : "grammar: <text>"          → outputs corrected sentence
# The prefix is just plain text — no special token needed, the model was trained on it
# Flan-T5 is instruction-tuned T5 — better at following natural language task descriptions
# SentencePiece tokenizer (not BPE or WordPiece) — uses unigram language model
#   No Ġ prefix for spaces; uses ▁ (U+2581) as a word-start marker instead

print("\nSTEP 6 — T5 / Flan-T5 task-prefix style")

t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
t5_model     = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
t5_model.eval()

print(f"  T5 tokenizer class : {type(t5_tokenizer).__name__}")  # T5Tokenizer
print(f"  T5 vocab size      : {t5_tokenizer.vocab_size:,}")    # 32,100
print(f"  T5 EOS token       : {t5_tokenizer.eos_token!r}")     # '</s>'
print(f"  T5 PAD token       : {t5_tokenizer.pad_token!r}")     # '<pad>'

# Task 1: Summarisation — "summarize:" prefix
t5_input_sum  = t5_tokenizer(
    "summarize: " + text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)
with torch.no_grad():
    t5_sum_ids = t5_model.generate(**t5_input_sum, max_new_tokens=100, num_beams=4)
t5_summary    = t5_tokenizer.decode(t5_sum_ids[0], skip_special_tokens=True)
print(f"\n  [summarize] → {t5_summary!r}")

# Task 2: Translation — "translate English to French:" prefix
t5_input_trans = t5_tokenizer(
    "translate English to French: The Amazon is the largest rainforest on Earth.",
    return_tensors="pt"
)
with torch.no_grad():
    t5_trans_ids = t5_model.generate(**t5_input_trans, max_new_tokens=50)
t5_translation  = t5_tokenizer.decode(t5_trans_ids[0], skip_special_tokens=True)
print(f"  [translate]  → {t5_translation!r}")
#   "L'Amazonie est la plus grande forêt tropicale de la Terre."

# Task 3: Grammar correction — "grammar:" prefix
t5_input_gram = t5_tokenizer(
    "grammar: She have went to the store yesterday.",
    return_tensors="pt"
)
with torch.no_grad():
    t5_gram_ids = t5_model.generate(**t5_input_gram, max_new_tokens=30)
t5_grammar      = t5_tokenizer.decode(t5_gram_ids[0], skip_special_tokens=True)
print(f"  [grammar]    → {t5_grammar!r}")
#   "She went to the store yesterday."


# ── BART vs T5 — MODEL COMPARISON ────────────────────────────────────────────
# Property              BART                            T5
# ─────────────────────────────────────────────────────────────────────────────
# Pre-training          Denoising (corrupt → reconstruct)  Span masking (text-to-text)
# Task specification    Fine-tune per task              Task prefix in the input string
# Tokenizer             BPE (RoBERTa-style, 50K vocab)  SentencePiece unigram (32K vocab)
# Special tokens        <s>, </s>, <pad>                </s>, <pad> — NO BOS by default
# Space marker          Ġ prefix                         ▁ prefix (SentencePiece)
# Architecture          Standard enc-dec transformer     Same, but relative position bias
# Good models           bart-large-cnn, bart-large-xsum  t5-base, flan-t5-large, t5-3b
# Best for              Summarisation, abs. extraction   Multi-task with prefix control
# ─────────────────────────────────────────────────────────────────────────────
# Both use the same .generate() API — swap model + tokenizer, everything else stays the same


# ── ENCODER vs DECODER ATTENTION TYPES ───────────────────────────────────────
# Component             Attention type          Attends to
# ─────────────────────────────────────────────────────────────────────────────
# Encoder self-attn     Bidirectional (full)    All encoder input tokens (both directions)
# Decoder self-attn     Causal (masked)         All PREVIOUS decoder output tokens only
# Decoder cross-attn    Full                    ALL encoder hidden states at every step
# ─────────────────────────────────────────────────────────────────────────────
# Cross-attention is the "bridge" — it is how the decoder reads the encoder's understanding
# Q (query) comes from the decoder hidden state  (what am I trying to generate next?)
# K, V (key/value) come from the encoder output  (what did the source text say?)


# ── OUTPUT DATACLASS FIELDS ───────────────────────────────────────────────────
# Seq2SeqLMOutput
#   .loss                    → scalar (only present when labels= passed)
#   .logits                  → shape [B, tgt_seq_len, vocab_size]  — decoder output
#   .past_key_values         → KV cache for decoder self-attn (speeds up generation)
#   .encoder_last_hidden_state → shape [B, src_seq_len, hidden_size]
#   .encoder_hidden_states   → tuple per layer (needs output_hidden_states=True)
#   .decoder_hidden_states   → tuple per layer (needs output_hidden_states=True)
#   .cross_attentions        → cross-attn weights [B, heads, tgt_len, src_len]
#                              useful for interpretability: see what source tokens
#                              the decoder attended to when generating each output token
#
# Passing labels= for fine-tuning loss:
#   labels = tokenizer(target_summary, return_tensors="pt").input_ids
#   labels[labels == tokenizer.pad_token_id] = -100   # ignore padding positions
#   outputs = model(**inputs, labels=labels)
#   outputs.loss  → cross-entropy averaged over all non-ignored decoder positions


# ── DRY RUN : text = "The Amazon rainforest is the world's largest..." ────────

# Stage 1 — Encoder tokenization (BPE, BART)
# tokenizer adds <s> at start, </s> at end automatically
# BPE splits on subword frequency — leading spaces absorbed into token (Ġ prefix)
#
# First few tokens:
# pos  raw        BPE piece    id
#  0   [auto]    <s>           0    ← BOS prepended by tokenizer
#  1   The       ĠThe          133
#  2   Amazon    ĠAmazon       35567
#  3   rainforest Ġrainforest  ???  ← may split: Ġrain + ##forest if rare
#  4   is        Ġis           16
#  ...
#  N   [auto]    </s>          2    ← EOS appended by tokenizer
#
# attention_mask: all 1s for real tokens — no padding in single-sequence case

# Stage 2 — Encoder forward pass (bidirectional, 12 layers)
# Input shape  : [1, src_len, 1024]  (after embedding + position encoding)
# All tokens attend to all tokens — no causal mask in encoder
# After 12 layers: encoder_hidden_states shape [1, src_len, 1024]
# Each position holds a contextualised vector summarising its token + full context
# "Amazon" at position 2 encodes: "large", "tropical", "rainforest", "lungs" all at once
# These 1024-dim vectors are what the decoder will cross-attend to at every step

# Stage 3 — Decoder initialisation (beam search, num_beams=4)
# 4 beams start as identical: decoder_input_ids = [[<s>], [<s>], [<s>], [<s>]]
# encoder_hidden_states are tiled 4× to match beam count: shape [4, src_len, 1024]
# cumulative log-prob for each beam = 0.0 initially

# Stage 4 — Decoder step 0 (generate first token)
# Decoder input  : [[<s>]] shape [4, 1]
# Self-attention : trivial — only one token, attends to itself
# Cross-attention: queries from <s> hidden state [1024] attend to all encoder positions
#                  K, V come from encoder_hidden_states [src_len, 1024]
#                  cross-attn output: decoder "reads" encoder to know what to say first
# LM head output : logits shape [4, 1, 50265]
# For each beam, take logits[:, -1, :] → top-K candidates
# Beam 1 selects: 'The'   log-prob = -0.21   cumulative = -0.21
# Beam 2 selects: 'Amazon' log-prob = -0.89  cumulative = -0.89
# Beam 3 selects: 'It'    log-prob = -1.03   cumulative = -1.03
# Beam 4 selects: 'More'  log-prob = -1.47   cumulative = -1.47
# Keep all 4 — they are the 4 globally best first tokens

# Stage 5 — Decoder step 1 (generate second token)
# Decoder input  : [[<s>, 'The'], [<s>, 'Amazon'], [<s>, 'It'], [<s>, 'More']]
# Self-attention : each beam attends causally to its own past tokens
# Cross-attention: same encoder states attended with new query from step-1 hidden state
# LM head: logits [4, 2, 50265] → use position -1 for next token
# Beam 1 ('The') → top candidate: 'ĠAmazon'  cumulative log-prob = -0.21 + -0.18 = -0.39
# Beam 2 ('Amazon') → top candidate: 'Ġrainforest' cumulative = -0.89 + -0.61 = -1.50
# Globally: Beam 1 still leads with "The Amazon..." at -0.39
# This continues for up to max_new_tokens=150 steps

# Stage 6 — Beam termination
# A beam terminates when it generates </s> (id=2) as its next token
# early_stopping=True → stop as soon as num_beams=4 complete sequences found
# no_repeat_ngram_size=3 → at each step, any token that would complete a trigram
#                           already in the hypothesis is set to -inf before sampling
# min_length=50 → </s> is blocked (set to -inf) for the first 50 decoder steps
# Final beams scored by total log-prob (optionally divided by length for fairness)
# Best scoring complete sequence selected → output_ids

# Stage 7 — Decode output_ids
# output_ids shape: [1, M]  where M is the length of the best beam
# tokenizer.decode(output_ids[0], skip_special_tokens=True)
#   strips <s> (id 0) and </s> (id 2)
#   converts BPE ids → string pieces
#   removes Ġ prefixes → real spaces
#   joins pieces → final summary string
#
# Final output:
#   "The Amazon rainforest covers over 5.5 million square kilometres and is home
#    to more than 30 million people. It has been described as the lungs of the
#    planet for its role in regulating the global climate."

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

————————————————————————————————————————————————————————————
STEP 1 — Tokenizer loaded
  Tokenizer class  : RobertaTokenizer
  Vocab size       : 50,265
  BOS token        : '<s>'
  EOS token        : '</s>'
  PAD token        : '<pad>'
  BOS id           : 0
  EOS id           : 2
  PAD id           : 1


...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

  T5 tokenizer class : T5Tokenizer
  T5 vocab size      : 32,100
  T5 EOS token       : '</s>'
  T5 PAD token       : '<pad>'

  [summarize] → "The Amazon rainforest is the world's largest tropical rainforest, covering over 5.5 million square kilometres."
  [translate]  → "L'Amazon est le plus grande ronstre sur l'Earth."
  [grammar]    → 'she went to the store yesterday.'
